# Agent 2 — Notebook 07
## Question Visual Cropping with Structural Parent/Sibling Boundaries

This notebook is a **rendering-only worker** for Notebook 05.

It does not perform topic detection, retrieval, ranking, question selection,
mark-scheme parsing, or assessment assembly.

Notebook 05 supplies the selected questions, source PDFs, page metadata and a
document-level question inventory. Notebook 07 keeps the original PDF as the
source of truth and applies the following structural crop model.

### v2.1 structural crop model

For every selected **child/subquestion**:

```text
selected question number
    ↓
derive direct structural parent
    01.1 -> 01
    08.1 -> 08
    02.3 -> 02
    ↓
find ALL direct child markers of that parent on the source PDF
    01.1
    01.2
    01.3
    ...
    ↓
DEPENDENCY / SHARED-CONTEXT CROP
    parent start OR required Figure/Table/Diagram/Flowchart caption
    ↓
    STOP immediately before FIRST child marker
    ↓
SELECTED-QUESTION CROP
    selected child marker
    ↓
    STOP immediately before NEXT sibling marker
```

For a selected **top-level question**, the renderer creates only the selected
question crop from that top-level marker to the next top-level question
boundary. It does **not** create a separate dependency crop when the figure or
other visual is already inside that top-level question.

This prevents three common failures:

1. shared parent context being repeated together with earlier sibling questions;
2. a selected child crop leaking into the next sibling;
3. a top-level question producing a duplicate dependency image.

Multi-page intervals are handled using the same boundaries: first page starts at
the verified structural start, middle pages retain their safe content area, and
the final page stops at the verified sibling/top-level marker.

### Safety rule

The notebook never OCRs or rewrites question content. It only renders regions
from the original source PDF.


## 1. Dependencies

Notebook 05 normally installs these dependencies already. This cell keeps
Notebook 07 independently executable for testing.

In [ ]:
# PERFORMANCE: PyMuPDF/pandas/numpy are expected to be installed once in the
# active project virtual environment. Do not reinstall them for every render.
print("Renderer dependencies loaded from the active virtual environment.")


## 2. Configuration and render-request contract

Notebook 05 supplies these environment variables before executing Notebook 07:

- `AGENT2_RENDER_REQUEST_PATH`
- `AGENT2_RENDER_MANIFEST_PATH`

The request contains absolute/relative source PDF paths and all question/page
metadata needed for rendering.

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import time
from datetime import datetime, timezone
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any

import fitz
import numpy as np
import pandas as pd
from IPython.display import Image as IPythonImage, display


NOTEBOOK7_RENDERER_VERSION = (
    "agent2-question-visual-renderer-v2.1.0-structural-parent-sibling-boundaries"
)

QUESTION_REGION_CROP_VERSION = (
    "agent2-notebook7-question-region-crop-v2.1.0-structural-sibling-safe"
)

MULTIPAGE_RENDER_VERSION = (
    "agent2-notebook7-multipage-render-v2.1.0-structural-intervals"
)

REQUEST_PATH = Path(
    os.environ.get(
        "AGENT2_RENDER_REQUEST_PATH",
        "",
    )
).expanduser()

MANIFEST_PATH = Path(
    os.environ.get(
        "AGENT2_RENDER_MANIFEST_PATH",
        "",
    )
).expanduser()

if not REQUEST_PATH.is_file():
    raise RuntimeError(
        "AGENT2_RENDER_REQUEST_PATH does not point to a render request JSON."
    )

if not str(MANIFEST_PATH):
    raise RuntimeError(
        "AGENT2_RENDER_MANIFEST_PATH is not configured."
    )

request_payload = json.loads(
    REQUEST_PATH.read_text(
        encoding="utf-8"
    )
)

PROJECT_ROOT = Path(
    request_payload.get(
        "project_root",
        Path.cwd(),
    )
).expanduser().resolve()

OUTPUT_DIR = Path(
    request_payload.get(
        "output_dir",
        PROJECT_ROOT / "OUTPUT",
    )
).expanduser().resolve()

RUN_TIMESTAMP = str(
    request_payload.get(
        "run_timestamp"
    )
    or datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
)

RENDER_DPI = int(
    request_payload.get(
        "render_dpi",
        180,
    )
)

DATABASE_PAGE_NUMBERS_ARE_ONE_BASED = bool(
    request_payload.get(
        "database_page_numbers_are_one_based",
        True,
    )
)

SOURCE_PAGE_SEARCH_RADIUS = int(
    request_payload.get(
        "source_page_search_radius",
        4,
    )
)


DEPENDENCY_RESOLUTION_VERSION = (
    "agent2-structural-parent-context-dependency-v2.1.0"
)

# Dependencies often sit on the immediately preceding source page, but use the
# same bounded radius already supplied by Notebook 05.
DEPENDENCY_PAGE_SEARCH_RADIUS = int(
    request_payload.get(
        "dependency_page_search_radius",
        SOURCE_PAGE_SEARCH_RADIUS,
    )
)


# Maximum number of parent/source pages used by the conservative dependency
# fallback when a standalone caption cannot be extracted from the PDF text.
MAX_DEPENDENCY_CONTAINER_FALLBACK_PAGES = int(
    request_payload.get(
        "max_dependency_container_fallback_pages",
        3,
    )
)

# Geometry tolerances are deliberately generic PDF-layout tolerances.
DEPENDENCY_CONTAINMENT_TOLERANCE_PT = 2.0
DEPENDENCY_CAPTION_TOP_MARGIN_PT = 20.0
DEPENDENCY_BEFORE_QUESTION_GAP_PT = 8.0

# Gap kept before the first source question/subquestion following a
# dependency region.
DEPENDENCY_SIBLING_BOUNDARY_GAP_PT = 8.0


# Question-number markers on AQA-style papers are normally in the left side of
# the page. The value is a layout proportion, not a hardcoded absolute x-value.
QUESTION_MARKER_LEFT_REGION_RATIO = float(
    request_payload.get(
        "question_marker_left_region_ratio",
        0.35,
    )
)

# Words whose vertical centres differ by at most this amount are treated as
# belonging to the same visible PDF line when reconstructing layout.
QUESTION_MARKER_LINE_Y_TOLERANCE_PT = float(
    request_payload.get(
        "question_marker_line_y_tolerance_pt",
        4.0,
    )
)


# Parent/container matching is stricter than ordinary selected-question
# matching because sibling subparts often have very similar text.
CONTAINER_ANCHOR_MIN_SCORE = float(
    request_payload.get(
        "container_anchor_min_score",
        0.90,
    )
)

DEPENDENCY_GEOMETRY_CAPTION_LOOKBACK_PT = float(
    request_payload.get(
        "dependency_geometry_caption_lookback_pt",
        240.0,
    )
)

DEPENDENCY_GEOMETRY_PADDING_PT = float(
    request_payload.get(
        "dependency_geometry_padding_pt",
        10.0,
    )
)

DEPENDENCY_MIN_GEOMETRY_AREA_PT2 = float(
    request_payload.get(
        "dependency_min_geometry_area_pt2",
        16.0,
    )
)

RENDERED_SIBLING_OVERLAP_TOLERANCE_PT = float(
    request_payload.get(
        "rendered_sibling_overlap_tolerance_pt",
        1.0,
    )
)

QUESTION_MARKER_BOTTOM_GAP_PT = float(
    request_payload.get(
        "question_marker_bottom_gap_pt",
        DEPENDENCY_SIBLING_BOUNDARY_GAP_PT,
    )
)

DISPLAY_RENDERED_IMAGES = bool(
    request_payload.get(
        "display_rendered_images",
        False,
    )
)

# PERFORMANCE: stay inside the database/inventory page neighbourhood during
# normal runtime. A complete-document fallback remains available explicitly
# for manual recovery/debugging.
ALLOW_FULL_PDF_FALLBACK = bool(
    request_payload.get(
        "allow_full_pdf_fallback",
        False,
    )
)

QUESTIONS = request_payload.get(
    "questions",
    [],
)

DOCUMENT_INVENTORY = request_payload.get(
    "document_inventory",
    [],
)

IMAGE_ROOT = (
    OUTPUT_DIR
    / "visual_question_crops"
    / RUN_TIMESTAMP
)

IMAGE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

MANIFEST_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "Notebook 07 renderer:",
    NOTEBOOK7_RENDERER_VERSION,
)
print(
    "Questions received:",
    len(QUESTIONS),
)
print(
    "Render DPI:",
    RENDER_DPI,
)
print(
    "Output image root:",
    IMAGE_ROOT,
)

## 3. Generic source-layout helpers

These helpers use PDF text coordinates, drawings and image regions. There are
no topic-specific keywords and no OCR.

In [ ]:

# ------------------------------------------------------------------
# PERFORMANCE: cache immutable page extraction results for this worker run.
# The renderer asks for the same page text/words/geometry many times while
# locating parents, children, dependencies and crop boundaries.
# ------------------------------------------------------------------
_PAGE_TEXT_CACHE: dict[tuple[str, int], str] = {}
_PAGE_WORDS_CACHE: dict[tuple[str, int], list[Any]] = {}
_PAGE_BLOCKS_CACHE: dict[tuple[str, int], list[Any]] = {}
_PAGE_DICT_CACHE: dict[tuple[str, int], dict[str, Any]] = {}
_PAGE_DRAWINGS_CACHE: dict[tuple[str, int], list[Any]] = {}
_PAGE_IMAGE_INFO_CACHE: dict[tuple[str, int], list[Any]] = {}
_PAGE_TEXT_LINE_RECORDS_CACHE: dict[tuple[str, int], list[dict[str, Any]]] = {}
_ANCHOR_RECT_CACHE: dict[
    tuple[tuple[str, int], str],
    tuple[fitz.Rect | None, str, float],
] = {}


def page_cache_key(
    page: fitz.Page,
) -> tuple[str, int]:
    parent = getattr(page, "parent", None)
    document_name = str(
        getattr(parent, "name", "")
        or f"document:{id(parent)}"
    )

    try:
        document_name = str(
            Path(document_name).expanduser().resolve()
        )
    except Exception:
        pass

    return (
        document_name.casefold(),
        int(page.number),
    )


def cached_page_text(
    page: fitz.Page,
) -> str:
    key = page_cache_key(page)
    if key not in _PAGE_TEXT_CACHE:
        _PAGE_TEXT_CACHE[key] = page.get_text(
            "text",
            sort=True,
        )
    return _PAGE_TEXT_CACHE[key]


def cached_page_words(
    page: fitz.Page,
) -> list[Any]:
    key = page_cache_key(page)
    if key not in _PAGE_WORDS_CACHE:
        _PAGE_WORDS_CACHE[key] = page.get_text(
            "words",
            sort=True,
        )
    return _PAGE_WORDS_CACHE[key]


def cached_page_blocks(
    page: fitz.Page,
) -> list[Any]:
    key = page_cache_key(page)
    if key not in _PAGE_BLOCKS_CACHE:
        _PAGE_BLOCKS_CACHE[key] = page.get_text(
            "blocks",
            sort=True,
        )
    return _PAGE_BLOCKS_CACHE[key]


def cached_page_dict(
    page: fitz.Page,
) -> dict[str, Any]:
    key = page_cache_key(page)
    if key not in _PAGE_DICT_CACHE:
        try:
            payload = page.get_text(
                "dict",
                sort=True,
            )
        except TypeError:
            payload = page.get_text(
                "dict"
            )
        _PAGE_DICT_CACHE[key] = payload
    return _PAGE_DICT_CACHE[key]


def cached_page_drawings(
    page: fitz.Page,
) -> list[Any]:
    key = page_cache_key(page)
    if key not in _PAGE_DRAWINGS_CACHE:
        try:
            value = page.get_drawings()
        except Exception:
            value = []
        _PAGE_DRAWINGS_CACHE[key] = value
    return _PAGE_DRAWINGS_CACHE[key]


def cached_page_image_info(
    page: fitz.Page,
) -> list[Any]:
    key = page_cache_key(page)
    if key not in _PAGE_IMAGE_INFO_CACHE:
        try:
            value = page.get_image_info(
                xrefs=True
            )
        except Exception:
            value = []
        _PAGE_IMAGE_INFO_CACHE[key] = value
    return _PAGE_IMAGE_INFO_CACHE[key]



SOURCE_MATCH_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by",
    "for", "from", "has", "in", "is", "it", "of",
    "on", "or", "that", "the", "this", "to", "using",
    "was", "were", "when", "which", "with", "you", "your",
}

EXPLICIT_DEPENDENCY_PATTERN = re.compile(
    r"\b(Figure|Table|Diagram|Flowchart)\s+(\d+[A-Za-z]?)\b",
    flags=re.IGNORECASE,
)

STRUCTURED_RESPONSE_PATTERN = re.compile(
    (
        r"\b(?:fill|complete|shade|circle|draw|mark|plot|show|write|tick)"
        r"\b.{0,120}\b(?:"
        r"array|table|grid|diagram|flowchart|box|boxes|"
        r"answer\s+(?:area|space|grid)|path|route|graph|"
        r"trace|sorting\s+step"
        r")"
    ),
    flags=re.IGNORECASE | re.DOTALL,
)


def json_safe(value: Any) -> Any:
    if value is None:
        return None

    if isinstance(
        value,
        (str, bool, int),
    ):
        return value

    if isinstance(value, float):
        return (
            value
            if math.isfinite(value)
            else None
        )

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple, set),
    ):
        return [
            json_safe(item)
            for item in value
        ]

    if hasattr(value, "item"):
        try:
            return json_safe(
                value.item()
            )
        except Exception:
            pass

    return str(value)


def output_relative_path(
    path: Path,
) -> str:
    try:
        return (
            path.resolve()
            .relative_to(
                OUTPUT_DIR.resolve()
            )
            .as_posix()
        )
    except ValueError:
        return (
            path.resolve()
            .as_posix()
        )


def resolve_local_pdf_path(
    raw_path: Any,
) -> Path | None:
    text_value = str(
        raw_path or ""
    ).strip()

    if not text_value:
        return None

    supplied = Path(
        text_value
    ).expanduser()

    candidates = [
        supplied,
        PROJECT_ROOT / supplied,
        Path.cwd() / supplied,
        PROJECT_ROOT.parent / supplied,
    ]

    seen = set()

    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except OSError:
            resolved = candidate

        key = str(resolved)

        if key in seen:
            continue

        seen.add(key)

        if (
            resolved.exists()
            and resolved.is_file()
        ):
            return resolved

    return None


def parse_page_numbers(
    value: Any,
) -> list[int]:
    if value is None:
        return []

    if isinstance(
        value,
        (list, tuple, set, np.ndarray),
    ):
        raw_values = list(value)

    elif isinstance(value, str):
        stripped = value.strip()

        if not stripped:
            return []

        try:
            decoded = json.loads(
                stripped
            )

            if isinstance(
                decoded,
                list,
            ):
                raw_values = decoded
            else:
                raw_values = re.findall(
                    r"\d+",
                    stripped,
                )

        except json.JSONDecodeError:
            raw_values = re.findall(
                r"\d+",
                stripped,
            )

    else:
        raw_values = [value]

    result = []

    for raw in raw_values:
        try:
            number = int(raw)
        except (
            TypeError,
            ValueError,
        ):
            continue

        if number > 0:
            result.append(number)

    return sorted(
        set(result)
    )


def assigned_pages_for_question(
    question: dict[str, Any],
) -> list[int]:
    pages = set(
        parse_page_numbers(
            question.get(
                "visual_page_numbers"
            )
        )
    )

    start = question.get(
        "page_start"
    )

    end = question.get(
        "page_end"
    )

    try:
        start_int = (
            int(start)
            if start is not None
            and not pd.isna(start)
            else None
        )
    except (
        TypeError,
        ValueError,
    ):
        start_int = None

    try:
        end_int = (
            int(end)
            if end is not None
            and not pd.isna(end)
            else start_int
        )
    except (
        TypeError,
        ValueError,
    ):
        end_int = start_int

    if start_int is not None:
        if end_int is None:
            end_int = start_int

        low = min(
            start_int,
            end_int,
        )
        high = max(
            start_int,
            end_int,
        )

        pages.update(
            range(
                low,
                high + 1,
            )
        )

    return sorted(pages)


def normalized_text(
    value: Any,
) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        " ",
        str(
            value or ""
        ).casefold(),
    ).strip()


def compact_text(
    value: Any,
) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(
            value or ""
        ).casefold(),
    )


def meaningful_tokens(
    value: Any,
) -> list[str]:
    return [
        token
        for token in normalized_text(
            value
        ).split()
        if (
            len(token) > 2
            and token
            not in SOURCE_MATCH_STOPWORDS
        )
    ]


def anchor_phrases(
    value: Any,
) -> list[str]:
    text_value = re.sub(
        r"\s+",
        " ",
        str(
            value or ""
        ).replace(
            "\r",
            " ",
        ).replace(
            "\n",
            " ",
        ),
    ).strip()

    tokens = text_value.split()

    if not tokens:
        return []

    phrases = []
    seen = set()

    for size in (
        16,
        14,
        12,
        10,
        8,
        6,
        5,
        4,
        3,
    ):
        if len(tokens) < size:
            continue

        # Beginning of the selected question is the strongest boundary.
        candidate = " ".join(
            tokens[:size]
        )

        key = normalized_text(
            candidate
        )

        if (
            key
            and key not in seen
        ):
            seen.add(key)
            phrases.append(
                candidate
            )

    if (
        len(tokens) < 3
        and text_value
    ):
        phrases.append(
            text_value
        )

    return phrases


def normalize_anchor_token(
    value: Any,
) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(
            value or ""
        ).casefold(),
    )


def exact_or_fuzzy_anchor_rect(
    page: fitz.Page,
    text_value: Any,
) -> tuple[
    fitz.Rect | None,
    str,
    float,
]:
    cache_key = (
        page_cache_key(page),
        str(text_value or ""),
    )

    cached = _ANCHOR_RECT_CACHE.get(
        cache_key
    )

    if cached is not None:
        return cached

    phrases = anchor_phrases(
        text_value
    )

    for phrase in phrases:
        try:
            rectangles = page.search_for(
                phrase
            )
        except Exception:
            rectangles = []

        if rectangles:
            result = (
                rectangles[0],
                "exact_text_anchor",
                1.0,
            )
            _ANCHOR_RECT_CACHE[
                cache_key
            ] = result
            return result

    words = cached_page_words(
        page
    )

    if not words:
        result = (
            None,
            "anchor_not_found",
            0.0,
        )
        _ANCHOR_RECT_CACHE[
            cache_key
        ] = result
        return result

    normalized_words = [
        normalize_anchor_token(
            word[4]
        )
        for word in words
    ]

    best_score = 0.0
    best_rect = None

    for phrase in phrases:
        target_tokens = [
            normalize_anchor_token(
                token
            )
            for token in phrase.split()
        ]

        target_tokens = [
            token
            for token in target_tokens
            if token
        ]

        if len(target_tokens) < 2:
            continue

        target_text = " ".join(
            target_tokens
        )

        target_size = len(
            target_tokens
        )

        candidate_sizes = sorted(
            {
                max(
                    2,
                    target_size - 1,
                ),
                target_size,
                target_size + 1,
            }
        )

        for window_size in candidate_sizes:
            if (
                window_size
                > len(normalized_words)
            ):
                continue

            for start_index in range(
                0,
                len(normalized_words)
                - window_size
                + 1,
            ):
                window_tokens = (
                    normalized_words[
                        start_index:
                        start_index + window_size
                    ]
                )

                score = SequenceMatcher(
                    None,
                    target_text,
                    " ".join(
                        window_tokens
                    ),
                ).ratio()

                if score <= best_score:
                    continue

                selected_words = words[
                    start_index:
                    start_index + window_size
                ]

                best_rect = fitz.Rect(
                    min(
                        word[0]
                        for word in selected_words
                    ),
                    min(
                        word[1]
                        for word in selected_words
                    ),
                    max(
                        word[2]
                        for word in selected_words
                    ),
                    max(
                        word[3]
                        for word in selected_words
                    ),
                )

                best_score = float(
                    score
                )

    if best_score >= 0.72:
        result = (
            best_rect,
            "fuzzy_word_anchor",
            best_score,
        )
    else:
        result = (
            None,
            "anchor_not_found",
            best_score,
        )

    _ANCHOR_RECT_CACHE[
        cache_key
    ] = result

    return result

def question_number_present(
    page_text: Any,
    question_number: Any,
) -> bool:
    compact_number = compact_text(
        question_number
    )

    if len(compact_number) < 2:
        return False

    return bool(
        compact_number
        in compact_text(
            page_text
        )
    )


def page_match_metrics(
    *,
    page: fitz.Page,
    question: dict[str, Any],
) -> dict[str, Any]:
    page_text = cached_page_text(
        page
    )

    question_text = str(
        question.get(
            "question_text"
        )
        or ""
    ).strip()

    question_tokens = list(
        dict.fromkeys(
            meaningful_tokens(
                question_text
            )
        )
    )

    page_tokens = set(
        meaningful_tokens(
            page_text
        )
    )

    overlap = sum(
        token in page_tokens
        for token in question_tokens
    )

    token_coverage = (
        overlap
        / len(question_tokens)
        if question_tokens
        else 0.0
    )

    (
        anchor_rect,
        anchor_method,
        anchor_score,
    ) = exact_or_fuzzy_anchor_rect(
        page,
        question_text,
    )

    number_match = (
        question_number_present(
            page_text,
            question.get(
                "question_number"
            ),
        )
    )

    # Short exam questions can be perfectly valid, so an anchor plus a
    # small amount of lexical coverage is enough for short prompts.
    short_question = bool(
        len(question_tokens)
        <= 4
    )

    valid_match = bool(
        anchor_rect is not None
        and (
            token_coverage >= 0.48
            or (
                short_question
                and token_coverage >= 0.30
            )
            or number_match
        )
    )

    combined_score = float(
        0.65
        * float(anchor_score)
        + 0.25
        * float(token_coverage)
        + 0.10
        * float(number_match)
    )

    return {
        "page_text": page_text,
        "anchor_rect": anchor_rect,
        "anchor_method": anchor_method,
        "anchor_score": float(
            anchor_score
        ),
        "token_coverage": float(
            token_coverage
        ),
        "question_number_match": bool(
            number_match
        ),
        "combined_score": (
            combined_score
        ),
        "valid_match": (
            valid_match
        ),
    }


def bounded_candidate_pages(
    *,
    assigned_pages: list[int],
    page_count: int,
) -> list[int]:
    if not assigned_pages:
        return []

    if DATABASE_PAGE_NUMBERS_ARE_ONE_BASED:
        minimum_valid = 1
        maximum_valid = page_count
    else:
        minimum_valid = 0
        maximum_valid = page_count - 1

    minimum_page = max(
        minimum_valid,
        min(assigned_pages)
        - SOURCE_PAGE_SEARCH_RADIUS,
    )

    maximum_page = min(
        maximum_valid,
        max(assigned_pages)
        + SOURCE_PAGE_SEARCH_RADIUS,
    )

    return list(
        range(
            minimum_page,
            maximum_page + 1,
        )
    )


def source_number_to_index(
    page_number: int,
) -> int:
    return (
        page_number - 1
        if DATABASE_PAGE_NUMBERS_ARE_ONE_BASED
        else page_number
    )


def content_bottom_y(
    page: fitz.Page,
    *,
    minimum_y: float,
) -> float:
    candidates = []

    for block in cached_page_blocks(
        page
    ):
        try:
            y0 = float(block[1])
            y1 = float(block[3])
        except Exception:
            continue

        if y1 > minimum_y:
            candidates.append(y1)

    drawings = cached_page_drawings(
        page
    )

    for drawing in drawings:
        rect = drawing.get("rect")
        if rect is None:
            continue
        if float(rect.y1) > minimum_y:
            candidates.append(
                float(rect.y1)
            )

    image_info = cached_page_image_info(
        page
    )

    for image in image_info:
        bbox = image.get("bbox")
        if bbox is None:
            continue

        rect = fitz.Rect(bbox)

        if float(rect.y1) > minimum_y:
            candidates.append(
                float(rect.y1)
            )

    if not candidates:
        return max(
            minimum_y + 90.0,
            float(page.rect.height)
            - 34.0,
        )

    return min(
        float(page.rect.height)
        - 24.0,
        max(candidates) + 12.0,
    )


def page_has_visual_structure(
    page: fitz.Page,
) -> bool:
    return bool(
        cached_page_drawings(page)
        or cached_page_image_info(page)
    )

def required_dependency_labels(
    *values: Any,
) -> list[str]:
    output = []
    seen = set()

    for value in values:
        for match in EXPLICIT_DEPENDENCY_PATTERN.finditer(
            str(value or "")
        ):
            label = (
                f"{match.group(1).lower()} "
                f"{match.group(2).lower()}"
            )

            if label not in seen:
                seen.add(label)
                output.append(label)

    return output


def structured_response_required(
    question_text: Any,
) -> bool:
    return bool(
        STRUCTURED_RESPONSE_PATTERN.search(
            str(
                question_text
                or ""
            )
        )
    )



def normalized_dependency_label(
    value: Any,
) -> str:
    match = EXPLICIT_DEPENDENCY_PATTERN.search(
        str(value or "")
    )

    if not match:
        return str(value or "").strip()

    return (
        f"{match.group(1).title()} "
        f"{match.group(2)}"
    )


def page_text_line_records(
    page: fitz.Page,
) -> list[dict[str, Any]]:
    """
    Return actual PDF text lines with their geometric rectangles.

    Cached per PDF page because dependency matching may ask for the same
    line geometry repeatedly while resolving several selected questions.
    """
    key = page_cache_key(
        page
    )

    cached = _PAGE_TEXT_LINE_RECORDS_CACHE.get(
        key
    )

    if cached is not None:
        return cached

    payload = cached_page_dict(
        page
    )

    records: list[dict[str, Any]] = []

    for block in payload.get(
        "blocks",
        [],
    ):
        if int(
            block.get(
                "type",
                0,
            )
        ) != 0:
            continue

        for line in block.get(
            "lines",
            [],
        ):
            spans = line.get(
                "spans",
                [],
            )

            text_value = " ".join(
                str(
                    span.get(
                        "text",
                        "",
                    )
                ).strip()
                for span in spans
                if str(
                    span.get(
                        "text",
                        "",
                    )
                ).strip()
            )

            text_value = re.sub(
                r"\s+",
                " ",
                text_value,
            ).strip()

            if not text_value:
                continue

            bbox = line.get(
                "bbox"
            )

            if bbox is None:
                span_boxes = [
                    span.get(
                        "bbox"
                    )
                    for span in spans
                    if span.get(
                        "bbox"
                    )
                    is not None
                ]

                if not span_boxes:
                    continue

                bbox = (
                    min(
                        item[0]
                        for item
                        in span_boxes
                    ),
                    min(
                        item[1]
                        for item
                        in span_boxes
                    ),
                    max(
                        item[2]
                        for item
                        in span_boxes
                    ),
                    max(
                        item[3]
                        for item
                        in span_boxes
                    ),
                )

            records.append(
                {
                    "text": text_value,
                    "rect": fitz.Rect(
                        bbox
                    ),
                }
            )

    _PAGE_TEXT_LINE_RECORDS_CACHE[
        key
    ] = records

    return records

def standalone_dependency_labels_from_line(
    line_text: Any,
) -> list[str]:
    """
    Return dependency labels only when the WHOLE line is caption-like.

    Accepted examples:
        Figure 2
        Table 4
        Figure 4   Figure 5

    Rejected examples:
        The algorithm in Figure 2 is a sorting algorithm.
        State the purpose of Figure 4.
        Figure 8 is used by the question below.

    This is the central protection against reference-text false positives.
    """
    text_value = re.sub(
        r"\s+",
        " ",
        str(
            line_text
            or ""
        ),
    ).strip()

    if not text_value:
        return []

    matches = list(
        EXPLICIT_DEPENDENCY_PATTERN.finditer(
            text_value
        )
    )

    if not matches:
        return []

    # Remove every dependency label. Only punctuation / separators may remain.
    residual_parts = []
    cursor = 0

    for match in matches:
        residual_parts.append(
            text_value[
                cursor:
                match.start()
            ]
        )

        cursor = match.end()

    residual_parts.append(
        text_value[
            cursor:
        ]
    )

    residual = " ".join(
        residual_parts
    )

    residual = re.sub(
        r"""[\s\.,:;|\-–—/\\()\[\]{}]+""",
        "",
        residual,
    )

    if residual:
        return []

    labels = []

    for match in matches:
        normalized = (
            f"{match.group(1).title()} "
            f"{match.group(2)}"
        )

        if normalized not in labels:
            labels.append(
                normalized
            )

    return labels


def exact_dependency_label_rects(
    page: fitz.Page,
    label: str,
) -> list[fitz.Rect]:
    """
    Locate only genuine standalone caption lines for a dependency.

    The function name is kept for Notebook 07 compatibility, but the v1.2
    semantics are intentionally stricter than PyMuPDF ``search_for``.
    """
    target = normalized_text(
        normalized_dependency_label(
            label
        )
    )

    rectangles: list[fitz.Rect] = []

    for record in page_text_line_records(
        page
    ):
        line_labels = (
            standalone_dependency_labels_from_line(
                record[
                    "text"
                ]
            )
        )

        normalized_line_labels = {
            normalized_text(
                item
            )
            for item in line_labels
        }

        if (
            target
            not in normalized_line_labels
        ):
            continue

        rect = record[
            "rect"
        ]

        if not any(
            (
                abs(
                    float(
                        existing.y0
                    )
                    - float(
                        rect.y0
                    )
                )
                <= 1.0
                and abs(
                    float(
                        existing.y1
                    )
                    - float(
                        rect.y1
                    )
                )
                <= 1.0
            )
            for existing
            in rectangles
        ):
            rectangles.append(
                rect
            )

    return sorted(
        rectangles,
        key=lambda rect: (
            float(
                rect.y0
            ),
            float(
                rect.x0
            ),
        ),
    )


# Guard the exact failure mode found in the diagnostic run.
assert standalone_dependency_labels_from_line(
    "Figure 2"
) == ["Figure 2"]

assert standalone_dependency_labels_from_line(
    "Figure 4 Figure 5"
) == [
    "Figure 4",
    "Figure 5",
]

assert not standalone_dependency_labels_from_line(
    "The algorithms in Figure 4 and Figure 5 have the same purpose."
)

assert not standalone_dependency_labels_from_line(
    "State the data type of sorted in the algorithm shown in Figure 2."
)

def dependency_candidate_pages(
    *,
    assigned_pages: list[int],
    matched_question_page: int | None,
    page_count: int,
) -> list[int]:
    seeds = set(
        assigned_pages
    )

    if matched_question_page is not None:
        seeds.add(
            int(matched_question_page)
        )

    if not seeds:
        return []

    if DATABASE_PAGE_NUMBERS_ARE_ONE_BASED:
        minimum_valid = 1
        maximum_valid = page_count
    else:
        minimum_valid = 0
        maximum_valid = page_count - 1

    minimum_page = max(
        minimum_valid,
        min(seeds)
        - DEPENDENCY_PAGE_SEARCH_RADIUS,
    )

    maximum_page = min(
        maximum_valid,
        max(seeds)
        + DEPENDENCY_PAGE_SEARCH_RADIUS,
    )

    return list(
        range(
            minimum_page,
            maximum_page + 1,
        )
    )


def rect_contains_y(
    rect: fitz.Rect,
    y: float,
    *,
    margin: float = 2.0,
) -> bool:
    return bool(
        float(rect.y0) - margin
        <= float(y)
        <= float(rect.y1) + margin
    )

def question_render_required(
    question: dict[str, Any],
) -> dict[str, Any]:
    labels = required_dependency_labels(
        question.get(
            "question_text"
        ),
        question.get(
            "context_text"
        ),
    )

    structured = structured_response_required(
        question.get(
            "question_text"
        )
    )

    assigned_pages = assigned_pages_for_question(
        question
    )

    multi_page = bool(
        len(assigned_pages) > 1
    )

    database_visual = bool(
        question.get(
            "has_visual",
            False,
        )
    )

    effective = bool(
        database_visual
        or labels
        or structured
        or multi_page
    )

    return {
        "database_visual": database_visual,
        "required_labels": labels,
        "structured_response_required": structured,
        "multi_page_question": multi_page,
        "effective_visual_required": effective,
    }

## 4. Question inventory and structural boundary helpers

The document inventory supplied by Notebook 05 provides the legitimate
question numbers and text. Notebook 07 uses those records to determine
**relationships** (parent, child, sibling), then physically locates the
corresponding markers on the original source PDF.

The v2.1 helpers added at the end of this section implement the required
parent/first-child/selected-child/next-sibling boundary model.


In [ ]:
inventory_df = pd.DataFrame(
    DOCUMENT_INVENTORY
)

if not inventory_df.empty:
    for column in (
        "question_id",
        "question_document_id",
    ):
        if column in inventory_df.columns:
            inventory_df[
                column
            ] = inventory_df[
                column
            ].astype(str)


def inventory_rows_for_document(
    document_id: Any,
) -> pd.DataFrame:
    if inventory_df.empty:
        return pd.DataFrame()

    document_value = str(
        document_id or ""
    )

    return inventory_df[
        inventory_df[
            "question_document_id"
        ].astype(str).eq(
            document_value
        )
    ].copy()


def inventory_row_pages(
    row: pd.Series,
) -> list[int]:
    pages = set(
        parse_page_numbers(
            row.get(
                "visual_page_numbers"
            )
        )
    )

    start = row.get(
        "page_start"
    )
    end = row.get(
        "page_end"
    )

    try:
        start_int = (
            int(start)
            if start is not None
            and not pd.isna(start)
            else None
        )
    except Exception:
        start_int = None

    try:
        end_int = (
            int(end)
            if end is not None
            and not pd.isna(end)
            else start_int
        )
    except Exception:
        end_int = start_int

    if start_int is not None:
        if end_int is None:
            end_int = start_int

        pages.update(
            range(
                min(
                    start_int,
                    end_int,
                ),
                max(
                    start_int,
                    end_int,
                )
                + 1,
            )
        )

    return sorted(pages)


def next_question_boundary_y(
    *,
    page: fitz.Page,
    question: dict[str, Any],
    source_page_number: int,
    current_top_y: float,
) -> float | None:
    document_rows = (
        inventory_rows_for_document(
            question.get(
                "question_document_id"
            )
        )
    )

    if document_rows.empty:
        return None

    current_id = str(
        question.get(
            "question_id"
        )
        or ""
    )

    boundaries = []

    for _, candidate in (
        document_rows.iterrows()
    ):
        candidate_id = str(
            candidate.get(
                "question_id"
            )
            or ""
        )

        if candidate_id == current_id:
            continue

        candidate_pages = (
            inventory_row_pages(
                candidate
            )
        )

        if (
            source_page_number
            not in candidate_pages
        ):
            continue

        (
            candidate_rect,
            _,
            candidate_score,
        ) = exact_or_fuzzy_anchor_rect(
            page,
            candidate.get(
                "question_text"
            ),
        )

        if (
            candidate_rect is None
            or candidate_score < 0.72
        ):
            continue

        if (
            float(candidate_rect.y0)
            <= float(current_top_y)
            + 10.0
        ):
            continue

        boundaries.append(
            float(
                candidate_rect.y0
            )
        )

    return (
        min(boundaries)
        if boundaries
        else None
    )



def question_number_parts(
    value: Any,
) -> tuple[str, ...]:
    raw_parts = [
        part
        for part in re.split(
            r"[^A-Za-z0-9]+",
            str(value or "").strip(),
        )
        if part
    ]

    return tuple(
        (
            str(int(part))
            if part.isdigit()
            else part.casefold()
        )
        for part in raw_parts
    )


def direct_parent_question_parts(
    value: Any,
) -> tuple[str, ...] | None:
    parts = question_number_parts(value)

    if len(parts) <= 1:
        return None

    return parts[:-1]


def is_direct_parent_question_number(
    *,
    candidate: Any,
    child: Any,
) -> bool:
    parent = direct_parent_question_parts(child)

    return bool(
        parent is not None
        and question_number_parts(candidate) == parent
    )


def direct_parent_inventory_row(
    *,
    question: dict[str, Any],
    source_page_number: int | None = None,
    candidate_pages: list[int] | None = None,
) -> pd.Series | None:
    child_number = str(
        question.get("question_number")
        or ""
    ).strip()

    if not child_number:
        return None

    rows = inventory_rows_for_document(
        question.get("question_document_id")
    )

    if rows.empty:
        return None

    page_filter = set(
        int(value)
        for value in (candidate_pages or [])
    )

    candidates = []

    for _, row in rows.iterrows():
        candidate_number = str(
            row.get("question_number")
            or ""
        ).strip()

        if not is_direct_parent_question_number(
            candidate=candidate_number,
            child=child_number,
        ):
            continue

        row_pages = inventory_row_pages(row)

        if (
            source_page_number is not None
            and int(source_page_number) not in row_pages
        ):
            continue

        if (
            page_filter
            and not (set(row_pages) & page_filter)
        ):
            continue

        page_distance = 0

        if (
            source_page_number is not None
            and row_pages
        ):
            page_distance = min(
                abs(int(source_page_number) - int(page))
                for page in row_pages
            )

        candidates.append(
            (
                -page_distance,
                len(
                    meaningful_tokens(
                        row.get("question_text")
                    )
                ),
                row,
            )
        )

    if not candidates:
        return None

    candidates.sort(
        key=lambda item: (
            item[0],
            item[1],
        ),
        reverse=True,
    )

    return candidates[0][2]


# Assessment-wide, one execution only.
# Page-level derived geometry caches for this renderer execution.
_PAGE_VISUAL_GEOMETRY_CACHE: dict[
    tuple[str, int],
    list[tuple[str, fitz.Rect]],
] = {}
_VISUAL_WORD_LINE_RECORDS_CACHE: dict[
    tuple[str, int],
    list[dict[str, Any]],
] = {}

GLOBAL_DEPENDENCY_ASSET_REGISTRY: dict[
    tuple[str, int, str],
    dict[str, Any],
] = {}

GLOBAL_RENDERED_QUESTION_RECT_REGISTRY: dict[
    tuple[str, int],
    list[dict[str, Any]],
] = {}


def normalized_pdf_registry_key(value: Any) -> str:
    try:
        return str(
            Path(str(value or "")).expanduser().resolve()
        ).casefold()
    except Exception:
        return str(value or "").casefold()


def dependency_asset_registry_key(
    *,
    source_pdf_key: str,
    source_page_number: int,
    label: str,
) -> tuple[str, int, str]:
    return (
        normalized_pdf_registry_key(source_pdf_key),
        int(source_page_number),
        normalized_text(
            normalized_dependency_label(label)
        ),
    )


def lookup_dependency_asset(
    *,
    source_pdf_key: str,
    candidate_pages: list[int],
    label: str,
) -> dict[str, Any] | None:
    for page_number in candidate_pages:
        asset = GLOBAL_DEPENDENCY_ASSET_REGISTRY.get(
            dependency_asset_registry_key(
                source_pdf_key=source_pdf_key,
                source_page_number=page_number,
                label=label,
            )
        )

        if asset is not None:
            return asset

    return None


def register_dependency_asset(
    *,
    source_pdf_key: str,
    source_page_number: int,
    labels: list[str],
    image_path: str,
    crop_rectangle: dict[str, Any],
) -> None:
    asset = {
        "image_path": image_path,
        "source_page_number": int(source_page_number),
        "labels": list(dict.fromkeys(labels)),
        "crop_rectangle": dict(crop_rectangle),
    }

    for label in labels:
        GLOBAL_DEPENDENCY_ASSET_REGISTRY[
            dependency_asset_registry_key(
                source_pdf_key=source_pdf_key,
                source_page_number=source_page_number,
                label=label,
            )
        ] = asset


def question_rect_registry_key(
    *,
    source_pdf_key: str,
    source_page_number: int,
) -> tuple[str, int]:
    return (
        normalized_pdf_registry_key(source_pdf_key),
        int(source_page_number),
    )


def stored_rect_to_fitz(
    value: dict[str, Any],
) -> fitz.Rect:
    return fitz.Rect(
        float(value["x0"]),
        float(value["y0"]),
        float(value["x1"]),
        float(value["y1"]),
    )


def rectangles_overlap_materially(
    first: fitz.Rect,
    second: fitz.Rect,
    *,
    tolerance: float = RENDERED_SIBLING_OVERLAP_TOLERANCE_PT,
) -> bool:
    intersection = first & second

    if intersection.is_empty:
        return False

    return bool(
        float(intersection.width) > tolerance
        and float(intersection.height) > tolerance
    )


def rect_overlaps_existing(
    *,
    candidate_rect: fitz.Rect,
    source_pdf_key: str,
    source_page_number: int,
    exclude_question_id: str | None = None,
) -> bool:
    key = question_rect_registry_key(
        source_pdf_key=source_pdf_key,
        source_page_number=source_page_number,
    )

    for item in GLOBAL_RENDERED_QUESTION_RECT_REGISTRY.get(
        key,
        [],
    ):
        if (
            exclude_question_id
            and str(item.get("question_id") or "")
            == str(exclude_question_id)
        ):
            continue

        if rectangles_overlap_materially(
            candidate_rect,
            stored_rect_to_fitz(item["rect"]),
        ):
            return True

    return False


def register_rendered_question_rectangles(
    *,
    source_pdf_key: str,
    question: dict[str, Any],
    rectangles: list[dict[str, Any]],
) -> None:
    for rectangle in rectangles:
        source_page = rectangle.get(
            "source_page_number"
        )

        if source_page is None:
            continue

        key = question_rect_registry_key(
            source_pdf_key=source_pdf_key,
            source_page_number=int(source_page),
        )

        GLOBAL_RENDERED_QUESTION_RECT_REGISTRY.setdefault(
            key,
            [],
        ).append(
            {
                "question_id": str(
                    question.get("question_id")
                    or ""
                ),
                "question_number": str(
                    question.get("question_number")
                    or ""
                ),
                "rect": {
                    "x0": float(rectangle["x0"]),
                    "y0": float(rectangle["y0"]),
                    "x1": float(rectangle["x1"]),
                    "y1": float(rectangle["y1"]),
                },
            }
        )


def page_visual_geometry_rects(
    page: fitz.Page,
) -> list[tuple[str, fitz.Rect]]:
    key = page_cache_key(
        page
    )

    cached = _PAGE_VISUAL_GEOMETRY_CACHE.get(
        key
    )

    if cached is not None:
        return cached

    output = []

    for drawing in cached_page_drawings(
        page
    ):
        rect = drawing.get("rect")

        if rect is None:
            continue

        rect = fitz.Rect(rect)

        if (
            float(rect.width * rect.height)
            >= DEPENDENCY_MIN_GEOMETRY_AREA_PT2
        ):
            output.append(
                ("drawing", rect)
            )

    for image in cached_page_image_info(
        page
    ):
        bbox = image.get("bbox")

        if bbox is None:
            continue

        rect = fitz.Rect(bbox)

        if (
            float(rect.width * rect.height)
            >= DEPENDENCY_MIN_GEOMETRY_AREA_PT2
        ):
            output.append(
                ("image", rect)
            )

    for block in cached_page_blocks(
        page
    ):
        try:
            rect = fitz.Rect(
                float(block[0]),
                float(block[1]),
                float(block[2]),
                float(block[3]),
            )
        except Exception:
            continue

        if (
            float(rect.width * rect.height)
            >= DEPENDENCY_MIN_GEOMETRY_AREA_PT2
        ):
            output.append(
                ("text_block", rect)
            )

    _PAGE_VISUAL_GEOMETRY_CACHE[
        key
    ] = output

    return output

def dependency_geometry_top_y(
    *,
    page: fitz.Page,
    caption_rects: list[fitz.Rect],
    hard_bottom_y: float,
    source_pdf_key: str,
    source_page_number: int,
) -> tuple[
    float,
    str,
    list[dict[str, Any]],
]:
    caption_top = min(
        float(rect.y0)
        for rect in caption_rects
    )

    minimum_candidate_y = max(
        18.0,
        caption_top
        - DEPENDENCY_GEOMETRY_CAPTION_LOOKBACK_PT,
    )

    accepted = []

    for kind, rect in page_visual_geometry_rects(
        page
    ):
        if (
            float(rect.y1) < minimum_candidate_y
            or float(rect.y0) >= float(hard_bottom_y)
        ):
            continue

        if rect_overlaps_existing(
            candidate_rect=rect,
            source_pdf_key=source_pdf_key,
            source_page_number=source_page_number,
        ):
            continue

        accepted.append(
            {
                "kind": kind,
                "x0": float(rect.x0),
                "y0": float(rect.y0),
                "x1": float(rect.x1),
                "y1": float(rect.y1),
            }
        )

    top_candidates = [
        caption_top
    ]

    # Drawings/images above a caption can expand the geometry upward.
    # Ordinary text above the caption is NOT used as parent positioning.
    for item in accepted:
        if (
            item["kind"] in {"drawing", "image"}
            and item["y1"] >= minimum_candidate_y
        ):
            top_candidates.append(
                item["y0"]
            )

    top_y = max(
        18.0,
        min(top_candidates)
        - DEPENDENCY_GEOMETRY_PADDING_PT,
    )

    return (
        top_y,
        "geometry_caption_visual_bounds",
        accepted,
    )


def overlap_safe_selected_question_anchor(
    *,
    page: fitz.Page,
    question: dict[str, Any],
    candidate_anchor: fitz.Rect,
    source_pdf_key: str,
    source_page_number: int,
) -> tuple[fitz.Rect, str]:
    if not rect_overlaps_existing(
        candidate_rect=candidate_anchor,
        source_pdf_key=source_pdf_key,
        source_page_number=source_page_number,
        exclude_question_id=str(
            question.get("question_id")
            or ""
        ),
    ):
        return (
            candidate_anchor,
            "selected_question_text_anchor",
        )

    marker_rect, marker_method = (
        question_number_marker_rect(
            page=page,
            question_number=(
                question.get("question_number")
            ),
        )
    )

    if (
        marker_rect is not None
        and not rect_overlaps_existing(
            candidate_rect=marker_rect,
            source_pdf_key=source_pdf_key,
            source_page_number=source_page_number,
            exclude_question_id=str(
                question.get("question_id")
                or ""
            ),
        )
    ):
        return (
            marker_rect,
            "selected_question_overlap_guard:"
            + marker_method,
        )

    return (
        candidate_anchor,
        "selected_question_overlap_guard_unresolved",
    )


assert is_direct_parent_question_number(
    candidate="01",
    child="01.2",
)

assert not is_direct_parent_question_number(
    candidate="01.1",
    child="01.2",
)

assert is_direct_parent_question_number(
    candidate="2.3",
    child="2.3.1",
)

def context_start_y(
    *,
    page: fitz.Page,
    question: dict[str, Any],
    question_anchor: fitz.Rect,
    source_pdf_key: str | None = None,
    source_page_number: int | None = None,
) -> tuple[float, str]:
    parent_row = direct_parent_inventory_row(
        question=question,
        source_page_number=source_page_number,
    )

    if parent_row is None:
        return (
            float(question_anchor.y0),
            "question_anchor_no_direct_parent",
        )

    parent_anchor, parent_method, parent_score = (
        exact_or_fuzzy_anchor_rect(
            page,
            parent_row.get("question_text"),
        )
    )

    if (
        parent_anchor is None
        or parent_score
        < CONTAINER_ANCHOR_MIN_SCORE
    ):
        return (
            float(question_anchor.y0),
            "question_anchor_parent_not_high_confidence",
        )

    if (
        source_pdf_key
        and source_page_number is not None
        and rect_overlaps_existing(
            candidate_rect=parent_anchor,
            source_pdf_key=source_pdf_key,
            source_page_number=source_page_number,
            exclude_question_id=str(
                question.get("question_id")
                or ""
            ),
        )
    ):
        return (
            float(question_anchor.y0),
            "question_anchor_parent_overlap_guard",
        )

    if (
        float(parent_anchor.y0)
        >= float(question_anchor.y0)
    ):
        return (
            float(question_anchor.y0),
            "question_anchor_parent_not_above",
        )

    lookback = (
        float(question_anchor.y0)
        - float(parent_anchor.y0)
    )

    if lookback > 360.0:
        return (
            float(question_anchor.y0),
            "question_anchor_parent_too_far",
        )

    return (
        float(parent_anchor.y0),
        "verified_direct_parent_anchor:"
        + parent_method,
    )



def render_clip_to_png(
    *,
    page: fitz.Page,
    clip: fitz.Rect,
    output_path: Path,
) -> None:
    scale = (
        float(RENDER_DPI)
        / 72.0
    )

    pixmap = page.get_pixmap(
        matrix=fitz.Matrix(
            scale,
            scale,
        ),
        clip=clip,
        alpha=False,
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    pixmap.save(
        str(output_path)
    )



def dependency_inventory_container(
    *,
    question: dict[str, Any],
    label: str,
    source_page_number: int | None = None,
    candidate_pages: list[int] | None = None,
) -> pd.Series | None:
    parent_row = direct_parent_inventory_row(
        question=question,
        source_page_number=source_page_number,
        candidate_pages=candidate_pages,
    )

    if parent_row is None:
        return None

    normalized_label = normalized_text(
        normalized_dependency_label(label)
    )

    if (
        normalized_label
        not in normalized_text(
            parent_row.get("question_text")
        )
    ):
        return None

    return parent_row



def inventory_visual_pages(
    row: pd.Series | None,
) -> list[int]:
    if row is None:
        return []

    return parse_page_numbers(
        row.get(
            "visual_page_numbers"
        )
    )


def selected_question_anchor_rect(
    *,
    page: fitz.Page,
    question: dict[str, Any],
) -> fitz.Rect | None:
    (
        anchor,
        _,
        score,
    ) = exact_or_fuzzy_anchor_rect(
        page,
        question.get(
            "question_text"
        ),
    )

    if (
        anchor is None
        or score < 0.72
    ):
        return None

    return anchor


def dependency_region_top_y(
    *,
    page: fitz.Page,
    caption_rects: list[fitz.Rect],
    container_row: pd.Series | None,
    hard_bottom_y: float | None = None,
    source_pdf_key: str = "",
    source_page_number: int = 0,
) -> tuple[
    float,
    str,
    list[dict[str, Any]],
]:
    # container_row is intentionally unused for standalone-caption positioning.
    if hard_bottom_y is None:
        hard_bottom_y = (
            float(page.rect.height)
            - 22.0
        )

    return dependency_geometry_top_y(
        page=page,
        caption_rects=caption_rects,
        hard_bottom_y=hard_bottom_y,
        source_pdf_key=source_pdf_key,
        source_page_number=source_page_number,
    )



def compact_question_number(
    value: Any,
) -> str:
    """
    Normalize a parsed question number for geometric matching.

    Examples:
        "01.1" -> "011"
        "08.1" -> "081"
        "02.3" -> "023"

    Only alphanumeric characters participate; punctuation/box separators in the
    PDF layout are intentionally ignored.
    """
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(
            value
            or ""
        ).casefold(),
    )


def visual_word_line_records(
    page: fitz.Page,
) -> list[dict[str, Any]]:
    """
    Reconstruct visible page lines from PyMuPDF word coordinates.

    This merges words even when question-number boxes and question text are in
    different PDF text blocks but visually share the same y-position.
    """
    key = page_cache_key(
        page
    )

    cached = _VISUAL_WORD_LINE_RECORDS_CACHE.get(
        key
    )

    if cached is not None:
        return cached

    words = cached_page_words(
        page
    )

    if not words:
        _VISUAL_WORD_LINE_RECORDS_CACHE[
            key
        ] = []
        return []

    prepared = []

    for word in words:
        try:
            x0 = float(
                word[0]
            )
            y0 = float(
                word[1]
            )
            x1 = float(
                word[2]
            )
            y1 = float(
                word[3]
            )
            text_value = str(
                word[4]
            ).strip()
        except Exception:
            continue

        if not text_value:
            continue

        prepared.append(
            {
                "x0": x0,
                "y0": y0,
                "x1": x1,
                "y1": y1,
                "cy": (
                    y0
                    + y1
                )
                / 2.0,
                "text": text_value,
            }
        )

    prepared.sort(
        key=lambda item: (
            item[
                "cy"
            ],
            item[
                "x0"
            ],
        )
    )

    rows: list[
        list[
            dict[str, Any]
        ]
    ] = []

    row_centres: list[
        float
    ] = []

    for word in prepared:
        best_index = None
        best_distance = None

        for index, centre in enumerate(
            row_centres
        ):
            distance = abs(
                float(
                    word[
                        "cy"
                    ]
                )
                - float(
                    centre
                )
            )

            if (
                distance
                <= QUESTION_MARKER_LINE_Y_TOLERANCE_PT
                and (
                    best_distance
                    is None
                    or distance
                    < best_distance
                )
            ):
                best_index = index
                best_distance = distance

        if best_index is None:
            rows.append(
                [
                    word
                ]
            )

            row_centres.append(
                float(
                    word[
                        "cy"
                    ]
                )
            )

        else:
            rows[
                best_index
            ].append(
                word
            )

            row_centres[
                best_index
            ] = sum(
                float(
                    item[
                        "cy"
                    ]
                )
                for item in rows[
                    best_index
                ]
            ) / len(
                rows[
                    best_index
                ]
            )

    records = []

    for row in rows:
        row.sort(
            key=lambda item: (
                item[
                    "x0"
                ]
            )
        )

        records.append(
            {
                "words": row,
                "text": " ".join(
                    item[
                        "text"
                    ]
                    for item in row
                ),
                "rect": fitz.Rect(
                    min(
                        item[
                            "x0"
                        ]
                        for item in row
                    ),
                    min(
                        item[
                            "y0"
                        ]
                        for item in row
                    ),
                    max(
                        item[
                            "x1"
                        ]
                        for item in row
                    ),
                    max(
                        item[
                            "y1"
                        ]
                        for item in row
                    ),
                ),
            }
        )

    records.sort(
        key=lambda item: (
            float(
                item[
                    "rect"
                ].y0
            ),
            float(
                item[
                    "rect"
                ].x0
            ),
        )
    )

    _VISUAL_WORD_LINE_RECORDS_CACHE[
        key
    ] = records

    return records


def left_marker_prefix_text(
    *,
    page: fitz.Page,
    line_record: dict[str, Any],
) -> tuple[
    str,
    fitz.Rect | None,
]:
    """
    Build a compact string from the left-hand words of a visible PDF line.

    Question-number boxes are normally positioned left of the question text.
    Restricting matching to a proportional left region prevents marks, code,
    page numbers and normal prose from becoming marker candidates.
    """
    x_limit = (
        float(
            page.rect.width
        )
        * QUESTION_MARKER_LEFT_REGION_RATIO
    )

    left_words = [
        word
        for word in line_record.get(
            "words",
            [],
        )
        if float(
            word[
                "x0"
            ]
        )
        <= x_limit
    ]

    if not left_words:
        return (
            "",
            None,
        )

    left_words.sort(
        key=lambda item: (
            item[
                "x0"
            ]
        )
    )

    compact = "".join(
        re.sub(
            r"[^a-z0-9]+",
            "",
            str(
                word[
                    "text"
                ]
            ).casefold(),
        )
        for word in left_words[
            :8
        ]
    )

    rect = fitz.Rect(
        min(
            word[
                "x0"
            ]
            for word in left_words
        ),
        min(
            word[
                "y0"
            ]
            for word in left_words
        ),
        max(
            word[
                "x1"
            ]
            for word in left_words
        ),
        max(
            word[
                "y1"
            ]
            for word in left_words
        ),
    )

    return (
        compact,
        rect,
    )


def question_number_marker_rect(
    *,
    page: fitz.Page,
    question_number: Any,
) -> tuple[
    fitz.Rect | None,
    str,
]:
    """
    Locate a genuine visible question-number marker from the PDF geometry.

    The legitimate question number comes from the source-document inventory;
    the PDF is used only to locate where that marker is physically drawn.
    """
    target = compact_question_number(
        question_number
    )

    if not target:
        return (
            None,
            "empty_question_number",
        )

    # Extremely short top-level values such as "1" are too ambiguous to match
    # geometrically. Their child/subquestion markers remain fully usable.
    if len(
        target
    ) < 2:
        return (
            None,
            "question_number_too_short",
        )

    candidates = []

    for line in visual_word_line_records(
        page
    ):
        (
            prefix,
            prefix_rect,
        ) = left_marker_prefix_text(
            page=page,
            line_record=line,
        )

        if (
            not prefix
            or prefix_rect is None
        ):
            continue

        # Exact prefix is preferred. A small suffix is allowed because the
        # first question-text word may still fall inside the left-region cutoff.
        if not prefix.startswith(
            target
        ):
            continue

        suffix = prefix[
            len(
                target
            ):
        ]

        # If many extra alphanumeric characters follow immediately, this was
        # probably prose/code rather than a boxed question number.
        if len(
            suffix
        ) > 12:
            continue

        candidates.append(
            (
                len(
                    suffix
                ),
                float(
                    prefix_rect.y0
                ),
                prefix_rect,
            )
        )

    if not candidates:
        return (
            None,
            "question_marker_not_found",
        )

    candidates.sort(
        key=lambda item: (
            item[0],
            item[1],
        )
    )

    return (
        candidates[0][2],
        "visual_question_number_marker",
    )


def first_layout_question_marker_after_y(
    *,
    page: fitz.Page,
    question: dict[str, Any],
    source_page_number: int,
    minimum_y: float,
) -> tuple[
    float | None,
    dict[str, Any] | None,
]:
    """
    Find the first source-layout question number physically below a dependency.

    This is stronger than question-text matching because it uses the boxed/
    left-margin question identifier itself.
    """
    document_rows = inventory_rows_for_document(
        question.get(
            "question_document_id"
        )
    )

    if document_rows.empty:
        return (
            None,
            None,
        )

    candidates = []

    seen_question_numbers = set()

    for _, row in document_rows.iterrows():
        row_pages = inventory_row_pages(
            row
        )

        if (
            int(
                source_page_number
            )
            not in row_pages
        ):
            continue

        question_number = str(
            row.get(
                "question_number"
            )
            or ""
        ).strip()

        if not question_number:
            continue

        compact_number = compact_question_number(
            question_number
        )

        if (
            not compact_number
            or compact_number
            in seen_question_numbers
        ):
            continue

        seen_question_numbers.add(
            compact_number
        )

        (
            marker_rect,
            marker_method,
        ) = question_number_marker_rect(
            page=page,
            question_number=question_number,
        )

        if marker_rect is None:
            continue

        marker_y = float(
            marker_rect.y0
        )

        if (
            marker_y
            <= float(
                minimum_y
            )
            + 10.0
        ):
            continue

        # Prefer the earliest geometric marker. At effectively the same y,
        # prefer a more specific (longer) parsed number such as 01.1 over 01.
        candidates.append(
            (
                marker_y,
                -len(
                    compact_number
                ),
                question_number,
                marker_method,
                marker_rect,
            )
        )

    if not candidates:
        return (
            None,
            None,
        )

    candidates.sort(
        key=lambda item: (
            item[0],
            item[1],
            item[2],
        )
    )

    (
        marker_y,
        _,
        question_number,
        marker_method,
        marker_rect,
    ) = candidates[0]

    return (
        marker_y,
        {
            "question_number": (
                question_number
            ),
            "anchor_method": (
                marker_method
            ),
            "marker_rect": {
                "x0": round(
                    float(
                        marker_rect.x0
                    ),
                    3,
                ),
                "y0": round(
                    float(
                        marker_rect.y0
                    ),
                    3,
                ),
                "x1": round(
                    float(
                        marker_rect.x1
                    ),
                    3,
                ),
                "y1": round(
                    float(
                        marker_rect.y1
                    ),
                    3,
                ),
            },
        },
    )



def first_inventory_question_anchor_after_y(
    *,
    page: fitz.Page,
    question: dict[str, Any],
    source_page_number: int,
    minimum_y: float,
) -> tuple[
    float | None,
    dict[str, Any] | None,
]:
    """
    Find the earliest genuine parsed question/subquestion anchor below
    ``minimum_y`` on the same source page.

    This is used to stop a dependency crop BEFORE the first following
    question, regardless of whether that following question is selected.
    """
    document_rows = inventory_rows_for_document(
        question.get(
            "question_document_id"
        )
    )

    if document_rows.empty:
        return (
            None,
            None,
        )

    candidates: list[
        tuple[
            float,
            int,
            str,
            str,
        ]
    ] = []

    for _, row in document_rows.iterrows():
        row_pages = inventory_row_pages(
            row
        )

        if (
            int(
                source_page_number
            )
            not in row_pages
        ):
            continue

        text_value = str(
            row.get(
                "question_text"
            )
            or ""
        ).strip()

        if not text_value:
            continue

        (
            anchor_rect,
            anchor_method,
            anchor_score,
        ) = exact_or_fuzzy_anchor_rect(
            page,
            text_value,
        )

        if (
            anchor_rect is None
            or anchor_score < 0.72
        ):
            continue

        anchor_y = float(
            anchor_rect.y0
        )

        if (
            anchor_y
            <= float(
                minimum_y
            )
            + 12.0
        ):
            continue

        question_number = str(
            row.get(
                "question_number"
            )
            or ""
        ).strip()

        # Prefer subquestions when anchors are effectively tied, because a
        # parent/context record may overlap the same source area.
        subquestion_priority = int(
            "." in question_number
        )

        candidates.append(
            (
                anchor_y,
                -subquestion_priority,
                question_number,
                anchor_method,
            )
        )

    if not candidates:
        return (
            None,
            None,
        )

    candidates.sort(
        key=lambda item: (
            item[0],
            item[1],
            item[2],
        )
    )

    (
        anchor_y,
        _,
        question_number,
        anchor_method,
    ) = candidates[0]

    return (
        float(
            anchor_y
        ),
        {
            "question_number": (
                question_number
            ),
            "anchor_method": (
                anchor_method
            ),
        },
    )



def dependency_region_bottom_y(
    *,
    page: fitz.Page,
    question: dict[str, Any],
    source_page_number: int,
    caption_rects: list[fitz.Rect],
    matched_question_page: int | None,
) -> tuple[float, str]:
    """
    Dependency bottom hierarchy:

    1. first REAL source-layout question-number marker after the dependency;
    2. first inventory question-text anchor after the dependency;
    3. conservative content bottom.

    The first rule prevents selected and unselected sibling questions from
    leaking into the dependency crop even when their text is fragmented across
    PDF blocks.
    """
    caption_bottom = max(
        float(
            rect.y1
        )
        for rect in caption_rects
    )

    (
        layout_marker_y,
        layout_marker_meta,
    ) = first_layout_question_marker_after_y(
        page=page,
        question=question,
        source_page_number=(
            source_page_number
        ),
        minimum_y=caption_bottom,
    )

    if layout_marker_y is not None:
        return (
            min(
                float(
                    page.rect.height
                )
                - 22.0,
                float(
                    layout_marker_y
                )
                - QUESTION_MARKER_BOTTOM_GAP_PT,
            ),
            (
                "before_first_layout_question_marker:"
                + str(
                    (
                        layout_marker_meta
                        or {}
                    ).get(
                        "question_number",
                        "",
                    )
                )
            ),
        )

    (
        first_question_y,
        first_question_meta,
    ) = first_inventory_question_anchor_after_y(
        page=page,
        question=question,
        source_page_number=(
            source_page_number
        ),
        minimum_y=caption_bottom,
    )

    if first_question_y is not None:
        return (
            min(
                float(
                    page.rect.height
                )
                - 22.0,
                float(
                    first_question_y
                )
                - DEPENDENCY_SIBLING_BOUNDARY_GAP_PT,
            ),
            (
                "before_first_inventory_question_anchor:"
                + str(
                    (
                        first_question_meta
                        or {}
                    ).get(
                        "question_number",
                        "",
                    )
                )
            ),
        )

    return (
        content_bottom_y(
            page,
            minimum_y=caption_bottom,
        ),
        "conservative_content_bottom",
    )



def rect_fully_contains(
    outer: fitz.Rect,
    inner: fitz.Rect,
    *,
    tolerance: float = DEPENDENCY_CONTAINMENT_TOLERANCE_PT,
) -> bool:
    return bool(
        float(
            outer.x0
        )
        <= float(
            inner.x0
        )
        + tolerance
        and float(
            outer.y0
        )
        <= float(
            inner.y0
        )
        + tolerance
        and float(
            outer.x1
        )
        >= float(
            inner.x1
        )
        - tolerance
        and float(
            outer.y1
        )
        >= float(
            inner.y1
        )
        - tolerance
    )


def dependency_region_is_fully_inside_question_crop(
    *,
    dependency_rect: fitz.Rect,
    source_page_number: int,
    question_crop_rectangles: list[dict[str, Any]],
) -> bool:
    """
    Complete-region containment replaces the old caption-intersection test.
    """
    for item in question_crop_rectangles:
        if int(
            item.get(
                "source_page_number"
            )
        ) != int(
            source_page_number
        ):
            continue

        question_rect = fitz.Rect(
            float(
                item[
                    "x0"
                ]
            ),
            float(
                item[
                    "y0"
                ]
            ),
            float(
                item[
                    "x1"
                ]
            ),
            float(
                item[
                    "y1"
                ]
            ),
        )

        if rect_fully_contains(
            question_rect,
            dependency_rect,
        ):
            return True

    return False


def conservative_container_fallback_pages(
    *,
    container_row: pd.Series,
    candidate_pages: list[int],
) -> list[int]:
    row_pages = inventory_row_pages(
        container_row
    )

    allowed = [
        page
        for page in row_pages
        if page in set(
            candidate_pages
        )
    ]

    if not allowed:
        allowed = list(
            row_pages
        )

    return allowed[
        :MAX_DEPENDENCY_CONTAINER_FALLBACK_PAGES
    ]


def render_dependency_page_region(
    *,
    page: fitz.Page,
    source_page_number: int,
    top_y: float,
    bottom_y: float,
    question_dir: Path,
    sequence: int,
    labels: list[str],
    method: str,
) -> tuple[
    str,
    dict[str, Any],
]:
    top_y = max(
        18.0,
        float(
            top_y
        ),
    )

    bottom_y = min(
        float(
            page.rect.height
        )
        - 22.0,
        float(
            bottom_y
        ),
    )

    if (
        bottom_y
        <= top_y
        + 70.0
    ):
        bottom_y = min(
            float(
                page.rect.height
            )
            - 22.0,
            top_y
            + 180.0,
        )

    clip = fitz.Rect(
        14.0,
        top_y,
        float(
            page.rect.width
        )
        - 14.0,
        bottom_y,
    )

    image_path = (
        question_dir
        / (
            f"dependency_{sequence:02d}_"
            f"source_{source_page_number}.png"
        )
    )

    render_clip_to_png(
        page=page,
        clip=clip,
        output_path=image_path,
    )

    relative_path = output_relative_path(
        image_path
    )

    rectangle_record = {
        "role": "dependency",
        "labels": labels,
        "source_page_number": int(
            source_page_number
        ),
        "x0": round(
            float(
                clip.x0
            ),
            3,
        ),
        "y0": round(
            float(
                clip.y0
            ),
            3,
        ),
        "x1": round(
            float(
                clip.x1
            ),
            3,
        ),
        "y1": round(
            float(
                clip.y1
            ),
            3,
        ),
        "height_pt": round(
            float(
                clip.height
            ),
            3,
        ),
        "method": method,
    }

    return (
        relative_path,
        rectangle_record,
    )


def render_exact_dependencies(
    *,
    document: fitz.Document,
    question: dict[str, Any],
    assigned_pages: list[int],
    matched_question_page: int | None,
    question_crop_rectangles: list[dict[str, Any]],
    question_dir: Path,
    required_labels: list[str],
    source_pdf_key: str,
) -> dict[str, Any]:
    """
    Generic v1.2 dependency resolver.

    Standalone caption lines are the primary truth. Parsed parent/source
    containers provide a conservative fallback only when PDF caption extraction
    cannot establish a standalone caption.
    """
    if not required_labels:
        return {
            "resolved_dependency_labels": [],
            "missing_dependency_labels": [],
            "dependency_source_pages": [],
            "dependency_crop_images": [],
            "dependency_crop_rectangles": [],
            "dependency_resolution_details": [],
            "dependency_render_status": (
                "not_required"
            ),
        }

    candidate_pages = dependency_candidate_pages(
        assigned_pages=assigned_pages,
        matched_question_page=matched_question_page,
        page_count=document.page_count,
    )

    normalized_required = [
        normalized_dependency_label(
            label
        )
        for label in required_labels
    ]

    # page -> label -> caption rectangles
    standalone_matches: dict[
        int,
        dict[
            str,
            list[fitz.Rect],
        ],
    ] = {}

    label_resolution: dict[
        str,
        dict[str, Any],
    ] = {}

    # -------------------------------------------------------------
    # A. Locate genuine standalone caption occurrences.
    # -------------------------------------------------------------
    for label in normalized_required:
        cached_asset = lookup_dependency_asset(
            source_pdf_key=source_pdf_key,
            candidate_pages=candidate_pages,
            label=label,
        )

        if cached_asset is not None:
            label_resolution[label] = {
                "mode": "global_asset_reuse",
                "asset": cached_asset,
            }
            continue

        candidates = []

        for source_page in candidate_pages:
            page_index = source_number_to_index(
                source_page
            )

            if (
                page_index < 0
                or page_index
                >= document.page_count
            ):
                continue

            page = document.load_page(
                page_index
            )

            caption_rects = exact_dependency_label_rects(
                page,
                label,
            )

            if not caption_rects:
                continue

            distance = (
                abs(
                    int(
                        source_page
                    )
                    - int(
                        matched_question_page
                    )
                )
                if matched_question_page
                is not None
                else 0
            )

            # When distances tie, a dependency before/on the selected question
            # is usually more plausible than a later repeated figure.
            after_question_penalty = int(
                matched_question_page
                is not None
                and int(
                    source_page
                )
                > int(
                    matched_question_page
                )
            )

            candidates.append(
                (
                    distance,
                    after_question_penalty,
                    int(
                        source_page
                    ),
                    caption_rects,
                )
            )

        if candidates:
            candidates.sort(
                key=lambda item: (
                    item[0],
                    item[1],
                    item[2],
                )
            )

            (
                _,
                _,
                chosen_page,
                chosen_rects,
            ) = candidates[0]

            standalone_matches.setdefault(
                chosen_page,
                {},
            )[label] = chosen_rects

            label_resolution[
                label
            ] = {
                "mode": (
                    "standalone_caption"
                ),
                "source_page": (
                    chosen_page
                ),
            }

            continue

        # ---------------------------------------------------------
        # B. Caption extraction fallback:
        #    use a parsed parent/source container, never an arbitrary
        #    drawing/image on the page.
        # ---------------------------------------------------------
        container = dependency_inventory_container(
            question=question,
            label=label,
            candidate_pages=candidate_pages,
        )

        if container is not None:
            fallback_pages = (
                conservative_container_fallback_pages(
                    container_row=container,
                    candidate_pages=candidate_pages,
                )
            )

            if fallback_pages:
                label_resolution[
                    label
                ] = {
                    "mode": (
                        "parent_container_fallback"
                    ),
                    "container_row": (
                        container
                    ),
                    "source_pages": (
                        fallback_pages
                    ),
                }

                continue

        label_resolution[
            label
        ] = {
            "mode": "unresolved",
        }

    dependency_images: list[str] = []
    dependency_pages: list[int] = []
    dependency_rectangles: list[
        dict[str, Any]
    ] = []
    details: list[
        dict[str, Any]
    ] = []

    resolved_labels: list[str] = []
    missing_labels: list[str] = []

    sequence = 0

    # -------------------------------------------------------------
    # B2. Assessment-wide dependency crop reuse.
    # -------------------------------------------------------------
    for label, resolution in label_resolution.items():
        if resolution.get("mode") != "global_asset_reuse":
            continue

        asset = resolution["asset"]

        image_path = str(
            asset.get("image_path")
            or ""
        )

        source_page = int(
            asset.get("source_page_number")
            or 0
        )

        if (
            image_path
            and image_path not in dependency_images
        ):
            dependency_images.append(
                image_path
            )

        if (
            source_page
            and source_page not in dependency_pages
        ):
            dependency_pages.append(
                source_page
            )

        crop_rectangle = asset.get(
            "crop_rectangle"
        )

        if (
            isinstance(crop_rectangle, dict)
            and crop_rectangle
            not in dependency_rectangles
        ):
            dependency_rectangles.append(
                crop_rectangle
            )

        if label not in resolved_labels:
            resolved_labels.append(label)

        details.append(
            {
                "label": label,
                "status": "global_dependency_asset_reused",
                "source_page": source_page,
                "dependency_image": image_path,
            }
        )


    # -------------------------------------------------------------
    # C. Render / verify standalone-caption regions page-by-page.
    # -------------------------------------------------------------
    for source_page in sorted(
        standalone_matches
    ):
        labels_on_page = standalone_matches[
            source_page
        ]

        page = document.load_page(
            source_number_to_index(
                source_page
            )
        )

        all_caption_rects = [
            rect
            for rects
            in labels_on_page.values()
            for rect
            in rects
        ]

        container_candidates = []

        for label in labels_on_page:
            container = dependency_inventory_container(
                question=question,
                label=label,
                source_page_number=source_page,
                candidate_pages=candidate_pages,
            )

            if container is not None:
                container_candidates.append(
                    container
                )

        container_row = (
            max(
                container_candidates,
                key=lambda row: len(
                    meaningful_tokens(
                        row.get(
                            "question_text"
                        )
                    )
                ),
            )
            if container_candidates
            else None
        )

        (
            bottom_y,
            bottom_method,
        ) = dependency_region_bottom_y(
            page=page,
            question=question,
            source_page_number=source_page,
            caption_rects=all_caption_rects,
            matched_question_page=(
                matched_question_page
            ),
        )

        (
            top_y,
            top_method,
            dependency_geometry_objects,
        ) = dependency_region_top_y(
            page=page,
            caption_rects=all_caption_rects,
            container_row=container_row,
            hard_bottom_y=bottom_y,
            source_pdf_key=source_pdf_key,
            source_page_number=source_page,
        )

        # The complete region must at minimum extend materially below the
        # standalone caption. This prevents a caption-only "dependency".
        minimum_bottom = (
            max(
                float(
                    rect.y1
                )
                for rect
                in all_caption_rects
            )
            + 90.0
        )

        bottom_y = min(
            float(
                page.rect.height
            )
            - 22.0,
            max(
                float(
                    bottom_y
                ),
                minimum_bottom,
            ),
        )

        dependency_rect = fitz.Rect(
            14.0,
            top_y,
            float(
                page.rect.width
            )
            - 14.0,
            bottom_y,
        )

        complete_inside = (
            dependency_region_is_fully_inside_question_crop(
                dependency_rect=(
                    dependency_rect
                ),
                source_page_number=(
                    source_page
                ),
                question_crop_rectangles=(
                    question_crop_rectangles
                ),
            )
        )

        page_labels = list(
            labels_on_page.keys()
        )

        for label in page_labels:
            if label not in resolved_labels:
                resolved_labels.append(
                    label
                )

        if complete_inside:
            for label in page_labels:
                details.append(
                    {
                        "label": label,
                        "status": (
                            "complete_dependency_region_inside_question_crop"
                        ),
                        "source_page": (
                            source_page
                        ),
                        "caption_match": (
                            "standalone_caption"
                        ),
                        "region_y0": round(
                            float(
                                dependency_rect.y0
                            ),
                            3,
                        ),
                        "region_y1": round(
                            float(
                                dependency_rect.y1
                            ),
                            3,
                        ),
                        "region_height_pt": round(
                            float(
                                dependency_rect.height
                            ),
                            3,
                        ),
                    }
                )

            continue

        sequence += 1

        (
            relative_path,
            rectangle_record,
        ) = render_dependency_page_region(
            page=page,
            source_page_number=(
                source_page
            ),
            top_y=top_y,
            bottom_y=bottom_y,
            question_dir=question_dir,
            sequence=sequence,
            labels=page_labels,
            method=(
                top_method
                + "+"
                + bottom_method
                + "+standalone_caption"
            ),
        )

        dependency_images.append(
            relative_path
        )

        if (
            source_page
            not in dependency_pages
        ):
            dependency_pages.append(
                source_page
            )

        rectangle_record[
            "geometry_object_count"
        ] = len(
            dependency_geometry_objects
        )

        dependency_rectangles.append(
            rectangle_record
        )

        register_dependency_asset(
            source_pdf_key=source_pdf_key,
            source_page_number=source_page,
            labels=page_labels,
            image_path=relative_path,
            crop_rectangle=rectangle_record,
        )

        for label in page_labels:
            details.append(
                {
                    "label": label,
                    "status": (
                        "separate_dependency_crop"
                    ),
                    "source_page": (
                        source_page
                    ),
                    "caption_match": (
                        "standalone_caption"
                    ),
                    "positioning": (
                        "geometry_first"
                    ),
                    "geometry_object_count": len(
                        dependency_geometry_objects
                    ),
                    "dependency_image": (
                        relative_path
                    ),
                    "region_height_pt": (
                        rectangle_record[
                            "height_pt"
                        ]
                    ),
                }
            )

    # -------------------------------------------------------------
    # D. Conservative parent/source-container fallback for labels whose
    #    standalone caption was not extractable.
    # -------------------------------------------------------------
    fallback_groups: dict[
        tuple[int, ...],
        dict[str, Any],
    ] = {}

    for label, resolution in (
        label_resolution.items()
    ):
        if (
            resolution.get(
                "mode"
            )
            != "parent_container_fallback"
        ):
            continue

        pages = tuple(
            int(
                page
            )
            for page in resolution[
                "source_pages"
            ]
        )

        fallback_groups.setdefault(
            pages,
            {
                "labels": [],
                "container_row": (
                    resolution[
                        "container_row"
                    ]
                ),
            },
        )[
            "labels"
        ].append(
            label
        )

    for pages, group in (
        fallback_groups.items()
    ):
        labels = list(
            dict.fromkeys(
                group[
                    "labels"
                ]
            )
        )

        container_row = group[
            "container_row"
        ]

        for label in labels:
            if label not in resolved_labels:
                resolved_labels.append(
                    label
                )

        for page_position, source_page in enumerate(
            pages,
            start=1,
        ):
            page_index = source_number_to_index(
                source_page
            )

            if (
                page_index < 0
                or page_index
                >= document.page_count
            ):
                continue

            page = document.load_page(
                page_index
            )

            if page_position == 1:
                (
                    container_anchor,
                    container_method,
                    container_score,
                ) = exact_or_fuzzy_anchor_rect(
                    page,
                    container_row.get(
                        "question_text"
                    ),
                )

                parent_anchor_safe = bool(
                    container_anchor
                    is not None
                    and container_score
                    >= CONTAINER_ANCHOR_MIN_SCORE
                )

                if (
                    parent_anchor_safe
                    and rect_overlaps_existing(
                        candidate_rect=container_anchor,
                        source_pdf_key=source_pdf_key,
                        source_page_number=source_page,
                        exclude_question_id=str(
                            question.get("question_id")
                            or ""
                        ),
                    )
                ):
                    parent_anchor_safe = False

                if parent_anchor_safe:
                    top_y = max(
                        18.0,
                        float(container_anchor.y0)
                        - 16.0,
                    )
                    top_method = (
                        "direct_parent_container_fallback:"
                        + container_method
                    )
                else:
                    top_y = 20.0
                    top_method = (
                        "direct_parent_overlap_safe_page_top"
                    )
            else:
                top_y = 20.0
                top_method = (
                    "parent_container_fallback_continuation"
                )

            (
                layout_marker_y,
                layout_marker_meta,
            ) = first_layout_question_marker_after_y(
                page=page,
                question=question,
                source_page_number=(
                    source_page
                ),
                minimum_y=(
                    top_y
                    + 40.0
                ),
            )

            if layout_marker_y is not None:
                bottom_y = (
                    float(
                        layout_marker_y
                    )
                    - QUESTION_MARKER_BOTTOM_GAP_PT
                )

                bottom_method = (
                    "before_first_layout_question_marker:"
                    + str(
                        (
                            layout_marker_meta
                            or {}
                        ).get(
                            "question_number",
                            "",
                        )
                    )
                )

            else:
                (
                    first_question_y,
                    first_question_meta,
                ) = first_inventory_question_anchor_after_y(
                    page=page,
                    question=question,
                    source_page_number=(
                        source_page
                    ),
                    minimum_y=(
                        top_y
                        + 40.0
                    ),
                )

                if first_question_y is not None:
                    bottom_y = (
                        float(
                            first_question_y
                        )
                        - DEPENDENCY_SIBLING_BOUNDARY_GAP_PT
                    )

                    bottom_method = (
                        "before_first_inventory_question_anchor:"
                        + str(
                            (
                                first_question_meta
                                or {}
                            ).get(
                                "question_number",
                                "",
                            )
                        )
                    )

                else:
                    bottom_y = content_bottom_y(
                        page,
                        minimum_y=top_y,
                    )

                    bottom_method = (
                        "conservative_content_bottom"
                    )

            fallback_rect = fitz.Rect(
                14.0,
                top_y,
                float(
                    page.rect.width
                )
                - 14.0,
                bottom_y,
            )

            complete_inside = (
                dependency_region_is_fully_inside_question_crop(
                    dependency_rect=(
                        fallback_rect
                    ),
                    source_page_number=(
                        source_page
                    ),
                    question_crop_rectangles=(
                        question_crop_rectangles
                    ),
                )
            )

            if complete_inside:
                continue

            sequence += 1

            (
                relative_path,
                rectangle_record,
            ) = render_dependency_page_region(
                page=page,
                source_page_number=(
                    source_page
                ),
                top_y=top_y,
                bottom_y=bottom_y,
                question_dir=question_dir,
                sequence=sequence,
                labels=labels,
                method=(
                    top_method
                    + "+"
                    + bottom_method
                ),
            )

            if (
                relative_path
                not in dependency_images
            ):
                dependency_images.append(
                    relative_path
                )

            if (
                source_page
                not in dependency_pages
            ):
                dependency_pages.append(
                    source_page
                )

            dependency_rectangles.append(
                rectangle_record
            )

            register_dependency_asset(
                source_pdf_key=source_pdf_key,
                source_page_number=source_page,
                labels=labels,
                image_path=relative_path,
                crop_rectangle=rectangle_record,
            )

        for label in labels:
            details.append(
                {
                    "label": label,
                    "status": (
                        "parent_container_fallback"
                    ),
                    "source_pages": list(
                        pages
                    ),
                    "caption_match": (
                        "standalone_caption_not_extractable"
                    ),
                }
            )

    # -------------------------------------------------------------
    # E. Mark truly unresolved dependencies.
    # -------------------------------------------------------------
    for label, resolution in (
        label_resolution.items()
    ):
        if (
            resolution.get(
                "mode"
            )
            != "unresolved"
        ):
            continue

        missing_labels.append(
            label
        )

        details.append(
            {
                "label": label,
                "status": (
                    "dependency_unresolved"
                ),
                "searched_pages": (
                    candidate_pages
                ),
                "reason": (
                    "no standalone caption and no parsed parent/source "
                    "container could be resolved"
                ),
            }
        )

    return {
        "resolved_dependency_labels": (
            list(
                dict.fromkeys(
                    resolved_labels
                )
            )
        ),
        "missing_dependency_labels": (
            list(
                dict.fromkeys(
                    missing_labels
                )
            )
        ),
        "dependency_source_pages": (
            sorted(
                set(
                    dependency_pages
                )
            )
        ),
        "dependency_crop_images": (
            dependency_images
        ),
        "dependency_crop_rectangles": (
            dependency_rectangles
        ),
        "dependency_resolution_details": (
            details
        ),
        "dependency_render_status": (
            "rendered"
            if not missing_labels
            else "unresolved"
        ),
    }

def safe_full_content_rect(
    page: fitz.Page,
) -> fitz.Rect:
    return fitz.Rect(
        14.0,
        20.0,
        float(page.rect.width) - 14.0,
        float(page.rect.height) - 24.0,
    )


# ================================================================
# v2.1 STRUCTURAL PARENT / SIBLING BOUNDARY MODEL
# ================================================================
#
# The crop boundary logic below is intentionally structural:
#
#   selected 01.2
#       -> parent 01
#       -> discover ALL direct children of 01 from the document inventory
#       -> physically locate those child markers on the source PDF
#       -> shared context: parent start / exact figure caption
#                          -> immediately before FIRST child marker
#       -> selected child: selected child marker
#                          -> immediately before NEXT sibling marker
#
# For a selected top-level question (for example 08), there is no separate
# dependency crop. Its own crop runs from the top-level marker to the next
# top-level question boundary, so any figure already inside 08 remains inside
# that one crop.


STRUCTURAL_BOUNDARY_GAP_PT = float(
    request_payload.get(
        "structural_boundary_gap_pt",
        8.0,
    )
)

STRUCTURAL_START_PADDING_PT = float(
    request_payload.get(
        "structural_start_padding_pt",
        8.0,
    )
)

STRUCTURAL_CAPTION_LOOKBACK_PAGES = int(
    request_payload.get(
        "structural_caption_lookback_pages",
        2,
    )
)


def raw_question_number_parts(
    value: Any,
) -> tuple[str, ...]:
    """
    Preserve the visible structural components of a question number.

    Examples
    --------
    "01.1" -> ("01", "1")
    "08.1" -> ("08", "1")
    "02.3" -> ("02", "3")
    "08"   -> ("08",)
    """
    return tuple(
        part
        for part in re.findall(
            r"[A-Za-z0-9]+",
            str(value or "").strip(),
        )
        if part
    )


def structural_parent_question_number(
    value: Any,
) -> str | None:
    """
    Return only the direct structural parent.

    01.1 -> 01
    08.1 -> 08
    02.3 -> 02
    """
    parts = raw_question_number_parts(
        value
    )

    if len(parts) <= 1:
        return None

    return ".".join(
        parts[:-1]
    )


def is_top_level_question_number(
    value: Any,
) -> bool:
    return len(
        raw_question_number_parts(
            value
        )
    ) == 1


def is_direct_child_of_parent(
    *,
    candidate: Any,
    parent: Any,
) -> bool:
    candidate_parts = question_number_parts(
        candidate
    )
    parent_parts = question_number_parts(
        parent
    )

    return bool(
        parent_parts
        and len(candidate_parts)
        == len(parent_parts) + 1
        and candidate_parts[
            :len(parent_parts)
        ]
        == parent_parts
    )


def inventory_row_for_selected_question(
    question: dict[str, Any],
) -> pd.Series | None:
    rows = inventory_rows_for_document(
        question.get(
            "question_document_id"
        )
    )

    if rows.empty:
        return None

    question_id = str(
        question.get(
            "question_id"
        )
        or ""
    )

    if (
        question_id
        and "question_id"
        in rows.columns
    ):
        id_matches = rows[
            rows[
                "question_id"
            ].astype(str).eq(
                question_id
            )
        ]

        if not id_matches.empty:
            return id_matches.iloc[0]

    selected_parts = question_number_parts(
        question.get(
            "question_number"
        )
    )

    if selected_parts:
        for _, row in rows.iterrows():
            if (
                question_number_parts(
                    row.get(
                        "question_number"
                    )
                )
                == selected_parts
            ):
                return row

    return None


def direct_child_inventory_rows(
    *,
    question: dict[str, Any],
    parent_number: str,
) -> list[pd.Series]:
    """
    Inventory tells us which structural siblings legitimately belong to the
    parent. The PDF still supplies the physical marker coordinates.
    """
    rows = inventory_rows_for_document(
        question.get(
            "question_document_id"
        )
    )

    if rows.empty:
        return []

    children = []

    for _, row in rows.iterrows():
        if is_direct_child_of_parent(
            candidate=row.get(
                "question_number"
            ),
            parent=parent_number,
        ):
            children.append(
                row
            )

    def child_sort_key(
        row: pd.Series,
    ) -> tuple[Any, ...]:
        parts = question_number_parts(
            row.get(
                "question_number"
            )
        )

        converted = []

        for part in parts:
            try:
                converted.append(
                    (
                        0,
                        int(part),
                    )
                )
            except Exception:
                converted.append(
                    (
                        1,
                        str(part),
                    )
                )

        return tuple(
            converted
        )

    children.sort(
        key=child_sort_key
    )

    return children


def all_valid_source_page_numbers(
    document: fitz.Document,
) -> list[int]:
    if DATABASE_PAGE_NUMBERS_ARE_ONE_BASED:
        return list(
            range(
                1,
                document.page_count
                + 1,
            )
        )

    return list(
        range(
            0,
            document.page_count,
        )
    )


def ordered_source_page_candidates(
    *,
    document: fitz.Document,
    preferred_pages: list[int] | None = None,
) -> list[int]:
    """
    Search preferred source pages first, then only a bounded neighbourhood.

    The previous implementation appended every page in the PDF, so one
    missing/weak anchor could trigger a full-document fuzzy scan for every
    selected question and sibling. Full-PDF fallback is now explicit.
    """
    valid_pages = all_valid_source_page_numbers(
        document
    )

    preferred = []

    for page_number in (
        preferred_pages
        or []
    ):
        try:
            page_number = int(
                page_number
            )
        except Exception:
            continue

        if (
            page_number
            in valid_pages
            and page_number
            not in preferred
        ):
            preferred.append(
                page_number
            )

    # If no page metadata exists, retain the original all-pages safety path;
    # there is no trustworthy bounded seed in that case.
    if not preferred:
        return valid_pages

    bounded = list(
        preferred
    )

    for radius in range(
        1,
        max(
            0,
            SOURCE_PAGE_SEARCH_RADIUS,
        )
        + 1,
    ):
        for page_number in preferred:
            for candidate in (
                page_number - radius,
                page_number + radius,
            ):
                if (
                    candidate
                    in valid_pages
                    and candidate
                    not in bounded
                ):
                    bounded.append(
                        candidate
                    )

    if not ALLOW_FULL_PDF_FALLBACK:
        return bounded

    return bounded + [
        page_number
        for page_number
        in valid_pages
        if page_number
        not in bounded
    ]

def structural_marker_candidates(
    *,
    page: fitz.Page,
    question_number: Any,
) -> list[dict[str, Any]]:
    """
    Return every plausible left-margin marker for a known question number.

    The target number comes from the parsed inventory; the PDF is used only
    for geometry. A text-anchor proximity check in locate_question_position()
    disambiguates top-level markers from similarly prefixed children.
    """
    target = compact_question_number(
        question_number
    )

    if not target:
        return []

    target_depth = len(
        question_number_parts(
            question_number
        )
    )

    candidates = []

    for line in visual_word_line_records(
        page
    ):
        (
            prefix,
            prefix_rect,
        ) = left_marker_prefix_text(
            page=page,
            line_record=line,
        )

        if (
            not prefix
            or prefix_rect is None
            or not prefix.startswith(
                target
            )
        ):
            continue

        suffix = prefix[
            len(target):
        ]

        # Too much trailing material almost certainly means normal prose/code,
        # not the boxed/left-margin source question marker.
        if len(
            suffix
        ) > 14:
            continue

        # A top-level target such as 01 should not automatically claim the
        # marker 01.1. Keep it as a low-priority candidate only; if a matching
        # parent text anchor is present on the same line it can still win.
        child_prefix_penalty = int(
            target_depth == 1
            and bool(
                suffix
            )
            and suffix[
                :1
            ].isdigit()
        )

        candidates.append(
            {
                "rect": prefix_rect,
                "suffix_length": len(
                    suffix
                ),
                "child_prefix_penalty": (
                    child_prefix_penalty
                ),
            }
        )

    return candidates


def locate_question_position(
    *,
    document: fitz.Document,
    question_number: Any,
    question_text: Any = None,
    preferred_pages: list[int] | None = None,
) -> dict[str, Any] | None:
    """
    Locate the physical source position of one legitimate inventory question.

    Priority:
      1. question-number marker on the same line as the question-text anchor;
      2. question-text anchor;
      3. question-number marker alone.

    This keeps boundaries structural while retaining a safe text fallback for
    PDFs whose question-number boxes are fragmented during text extraction.
    """
    text_value = str(
        question_text
        or ""
    ).strip()

    text_fallbacks = []
    marker_fallbacks = []

    for page_rank, source_page in enumerate(
        ordered_source_page_candidates(
            document=document,
            preferred_pages=(
                preferred_pages
                or []
            ),
        )
    ):
        page_index = source_number_to_index(
            source_page
        )

        if (
            page_index < 0
            or page_index
            >= document.page_count
        ):
            continue

        page = document.load_page(
            page_index
        )

        (
            text_rect,
            text_method,
            text_score,
        ) = (
            exact_or_fuzzy_anchor_rect(
                page,
                text_value,
            )
            if text_value
            else (
                None,
                "no_question_text",
                0.0,
            )
        )

        marker_candidates = (
            structural_marker_candidates(
                page=page,
                question_number=question_number,
            )
        )

        if (
            text_rect is not None
            and marker_candidates
        ):
            ranked_markers = sorted(
                marker_candidates,
                key=lambda item: (
                    abs(
                        float(
                            item[
                                "rect"
                            ].y0
                        )
                        - float(
                            text_rect.y0
                        )
                    ),
                    int(
                        item[
                            "child_prefix_penalty"
                        ]
                    ),
                    int(
                        item[
                            "suffix_length"
                        ]
                    ),
                    float(
                        item[
                            "rect"
                        ].y0
                    ),
                ),
            )

            best_marker = ranked_markers[
                0
            ]

            vertical_distance = abs(
                float(
                    best_marker[
                        "rect"
                    ].y0
                )
                - float(
                    text_rect.y0
                )
            )

            if vertical_distance <= 42.0:
                return {
                    "question_number": str(
                        question_number
                        or ""
                    ),
                    "source_page_number": int(
                        source_page
                    ),
                    "rect": best_marker[
                        "rect"
                    ],
                    "anchor_method": (
                        "structural_question_marker+"
                        + text_method
                    ),
                    "anchor_score": float(
                        text_score
                    ),
                }

        if text_rect is not None:
            text_fallbacks.append(
                (
                    int(
                        page_rank
                    ),
                    -float(
                        text_score
                    ),
                    int(
                        source_page
                    ),
                    {
                        "question_number": str(
                            question_number
                            or ""
                        ),
                        "source_page_number": int(
                            source_page
                        ),
                        "rect": text_rect,
                        "anchor_method": (
                            "question_text_anchor_fallback:"
                            + text_method
                        ),
                        "anchor_score": float(
                            text_score
                        ),
                    },
                )
            )

        for marker in marker_candidates:
            marker_fallbacks.append(
                (
                    int(
                        marker[
                            "child_prefix_penalty"
                        ]
                    ),
                    int(
                        marker[
                            "suffix_length"
                        ]
                    ),
                    int(
                        page_rank
                    ),
                    int(
                        source_page
                    ),
                    float(
                        marker[
                            "rect"
                        ].y0
                    ),
                    {
                        "question_number": str(
                            question_number
                            or ""
                        ),
                        "source_page_number": int(
                            source_page
                        ),
                        "rect": marker[
                            "rect"
                        ],
                        "anchor_method": (
                            "structural_question_marker_only"
                        ),
                        "anchor_score": 0.0,
                    },
                )
            )

    if text_fallbacks:
        text_fallbacks.sort(
            key=lambda item: (
                item[0],
                item[1],
                item[2],
            )
        )
        return text_fallbacks[0][3]

    if marker_fallbacks:
        marker_fallbacks.sort(
            key=lambda item: (
                item[0],
                item[1],
                item[2],
                item[3],
                item[4],
            )
        )
        return marker_fallbacks[0][5]

    return None


def source_position_key(
    position: dict[str, Any],
) -> tuple[int, float]:
    return (
        int(
            position[
                "source_page_number"
            ]
        ),
        float(
            position[
                "rect"
            ].y0
        ),
    )


def position_is_before(
    first: dict[str, Any],
    second: dict[str, Any],
) -> bool:
    return source_position_key(
        first
    ) < source_position_key(
        second
    )


def locate_structural_child_markers(
    *,
    document: fitz.Document,
    question: dict[str, Any],
    parent_number: str,
) -> list[dict[str, Any]]:
    """
    Find ALL direct child markers of one structural parent.

    Example: parent 01 -> 01.1, 01.2, 01.3 ... physically located on the PDF.
    """
    hits = []
    seen_numbers = set()

    for row in direct_child_inventory_rows(
        question=question,
        parent_number=parent_number,
    ):
        question_number = str(
            row.get(
                "question_number"
            )
            or ""
        ).strip()

        normalized_number = (
            question_number_parts(
                question_number
            )
        )

        if (
            not normalized_number
            or normalized_number
            in seen_numbers
        ):
            continue

        preferred_pages = (
            inventory_row_pages(
                row
            )
        )

        hit = locate_question_position(
            document=document,
            question_number=question_number,
            question_text=row.get(
                "question_text"
            ),
            preferred_pages=preferred_pages,
        )

        if hit is None:
            continue

        seen_numbers.add(
            normalized_number
        )
        hits.append(
            hit
        )

    # Ensure the selected child is represented even if it was absent from the
    # inventory slice supplied by Notebook 05.
    selected_number = str(
        question.get(
            "question_number"
        )
        or ""
    ).strip()

    selected_parts = question_number_parts(
        selected_number
    )

    if (
        selected_parts
        and is_direct_child_of_parent(
            candidate=selected_number,
            parent=parent_number,
        )
        and selected_parts
        not in seen_numbers
    ):
        selected_hit = locate_question_position(
            document=document,
            question_number=selected_number,
            question_text=question.get(
                "question_text"
            ),
            preferred_pages=(
                assigned_pages_for_question(
                    question
                )
            ),
        )

        if selected_hit is not None:
            hits.append(
                selected_hit
            )

    hits.sort(
        key=source_position_key
    )

    return hits


def locate_dependency_caption_hits(
    *,
    document: fitz.Document,
    labels: list[str],
    candidate_pages: list[int],
) -> list[dict[str, Any]]:
    hits = []

    for source_page in candidate_pages:
        page_index = source_number_to_index(
            source_page
        )

        if (
            page_index < 0
            or page_index
            >= document.page_count
        ):
            continue

        page = document.load_page(
            page_index
        )

        for label in labels:
            for rect in exact_dependency_label_rects(
                page,
                label,
            ):
                hits.append(
                    {
                        "label": (
                            normalized_dependency_label(
                                label
                            )
                        ),
                        "source_page_number": int(
                            source_page
                        ),
                        "rect": rect,
                        "anchor_method": (
                            "standalone_dependency_caption"
                        ),
                        "anchor_score": 1.0,
                    }
                )

    hits.sort(
        key=source_position_key
    )

    return hits


def locate_parent_context_start(
    *,
    document: fitz.Document,
    question: dict[str, Any],
    parent_number: str,
    first_child: dict[str, Any],
    required_labels: list[str],
) -> tuple[
    dict[str, Any] | None,
    dict[str, Any],
]:
    """
    Shared context starts at the structural parent when possible. If a required
    Figure/Table/Diagram/Flowchart caption begins earlier, include that caption
    too. If the parent anchor cannot be resolved, the exact caption is the
    fallback start.
    """
    parent_row = direct_parent_inventory_row(
        question=question,
        candidate_pages=None,
    )

    parent_hit = None

    if parent_row is not None:
        parent_hit = locate_question_position(
            document=document,
            question_number=parent_row.get(
                "question_number"
            )
            or parent_number,
            question_text=parent_row.get(
                "question_text"
            ),
            preferred_pages=(
                inventory_row_pages(
                    parent_row
                )
            ),
        )

    valid_pages = all_valid_source_page_numbers(
        document
    )

    first_child_page = int(
        first_child[
            "source_page_number"
        ]
    )

    first_valid = min(
        valid_pages
    ) if valid_pages else first_child_page

    caption_start_page = max(
        first_valid,
        first_child_page
        - STRUCTURAL_CAPTION_LOOKBACK_PAGES,
    )

    if parent_hit is not None:
        caption_start_page = max(
            first_valid,
            min(
                int(
                    parent_hit[
                        "source_page_number"
                    ]
                ),
                caption_start_page,
            ),
        )

    caption_pages = [
        page_number
        for page_number in valid_pages
        if (
            caption_start_page
            <= page_number
            <= first_child_page
        )
    ]

    caption_hits = locate_dependency_caption_hits(
        document=document,
        labels=required_labels,
        candidate_pages=caption_pages,
    )

    caption_hits = [
        hit
        for hit in caption_hits
        if position_is_before(
            hit,
            first_child,
        )
    ]

    caption_hit = (
        caption_hits[
            0
        ]
        if caption_hits
        else None
    )

    candidates = [
        candidate
        for candidate in (
            parent_hit,
            caption_hit,
        )
        if (
            candidate is not None
            and position_is_before(
                candidate,
                first_child,
            )
        )
    ]

    if candidates:
        start_hit = min(
            candidates,
            key=source_position_key,
        )

        return (
            start_hit,
            {
                "parent_number": (
                    parent_number
                ),
                "parent_anchor_found": bool(
                    parent_hit is not None
                ),
                "caption_anchor_found": bool(
                    caption_hit is not None
                ),
                "start_method": (
                    "structural_parent_start"
                    if (
                        parent_hit is not None
                        and start_hit is parent_hit
                    )
                    else (
                        "standalone_dependency_caption_start"
                    )
                ),
            },
        )

    # Last safe structural fallback: retain only the visible context on the
    # first-child page above the first child marker. This never crosses into a
    # sibling question because the bottom boundary is still the first child.
    page = document.load_page(
        source_number_to_index(
            first_child_page
        )
    )

    fallback_rect = fitz.Rect(
        14.0,
        20.0,
        float(
            page.rect.width
        )
        - 14.0,
        20.0,
    )

    return (
        {
            "question_number": (
                parent_number
            ),
            "source_page_number": (
                first_child_page
            ),
            "rect": fallback_rect,
            "anchor_method": (
                "same_page_context_top_fallback"
            ),
            "anchor_score": 0.0,
        },
        {
            "parent_number": (
                parent_number
            ),
            "parent_anchor_found": False,
            "caption_anchor_found": False,
            "start_method": (
                "same_page_context_top_fallback"
            ),
        },
    )


def locate_next_top_level_boundary(
    *,
    document: fitz.Document,
    question: dict[str, Any],
    after_position: dict[str, Any],
) -> dict[str, Any] | None:
    rows = inventory_rows_for_document(
        question.get(
            "question_document_id"
        )
    )

    if rows.empty:
        return None

    candidates = []
    seen_numbers = set()

    for _, row in rows.iterrows():
        number = str(
            row.get(
                "question_number"
            )
            or ""
        ).strip()

        parts = question_number_parts(
            number
        )

        if (
            len(parts) != 1
            or parts
            in seen_numbers
        ):
            continue

        seen_numbers.add(
            parts
        )

        hit = locate_question_position(
            document=document,
            question_number=number,
            question_text=row.get(
                "question_text"
            ),
            preferred_pages=(
                inventory_row_pages(
                    row
                )
            ),
        )

        if (
            hit is not None
            and position_is_before(
                after_position,
                hit,
            )
        ):
            candidates.append(
                hit
            )

    return (
        min(
            candidates,
            key=source_position_key,
        )
        if candidates
        else None
    )


def source_pages_between(
    *,
    document: fitz.Document,
    first_page: int,
    last_page: int,
) -> list[int]:
    valid_pages = all_valid_source_page_numbers(
        document
    )

    low = min(
        int(
            first_page
        ),
        int(
            last_page
        ),
    )
    high = max(
        int(
            first_page
        ),
        int(
            last_page
        ),
    )

    return [
        page_number
        for page_number in valid_pages
        if low
        <= page_number
        <= high
    ]


def render_structural_interval(
    *,
    document: fitz.Document,
    start_position: dict[str, Any],
    end_position: dict[str, Any] | None,
    fallback_last_page: int | None,
    question_dir: Path,
    filename_prefix: str,
    role: str,
    method: str,
) -> tuple[
    list[str],
    list[dict[str, Any]],
]:
    """
    Render one structural interval over one or many source pages.

    First page starts at the selected structural marker/context start.
    Final page stops immediately before the supplied sibling/top-level marker.
    Middle pages keep the full safe content region.
    """
    first_page = int(
        start_position[
            "source_page_number"
        ]
    )

    if end_position is not None:
        last_page = int(
            end_position[
                "source_page_number"
            ]
        )
    elif fallback_last_page is not None:
        last_page = max(
            first_page,
            int(
                fallback_last_page
            ),
        )
    else:
        last_page = first_page

    if (
        end_position is not None
        and source_position_key(
            end_position
        )
        <= source_position_key(
            start_position
        )
    ):
        end_position = None
        last_page = max(
            first_page,
            int(
                fallback_last_page
                or first_page
            ),
        )

    page_numbers = source_pages_between(
        document=document,
        first_page=first_page,
        last_page=last_page,
    )

    image_paths = []
    rectangles = []

    for sequence, source_page in enumerate(
        page_numbers,
        start=1,
    ):
        page = document.load_page(
            source_number_to_index(
                source_page
            )
        )

        is_first = bool(
            source_page
            == first_page
        )
        is_last = bool(
            source_page
            == last_page
        )

        if is_first:
            top_y = max(
                18.0,
                float(
                    start_position[
                        "rect"
                    ].y0
                )
                - STRUCTURAL_START_PADDING_PT,
            )
        else:
            top_y = 20.0

        if (
            is_last
            and end_position is not None
            and int(
                end_position[
                    "source_page_number"
                ]
            )
            == source_page
        ):
            bottom_y = min(
                float(
                    page.rect.height
                )
                - 22.0,
                float(
                    end_position[
                        "rect"
                    ].y0
                )
                - STRUCTURAL_BOUNDARY_GAP_PT,
            )
            boundary_method = (
                "stop_before:"
                + str(
                    end_position.get(
                        "question_number"
                    )
                    or "boundary"
                )
            )
        elif is_last:
            bottom_y = content_bottom_y(
                page,
                minimum_y=top_y,
            )
            boundary_method = (
                "content_bottom_fallback"
            )
        else:
            bottom_y = (
                float(
                    page.rect.height
                )
                - 24.0
            )
            boundary_method = (
                "multi_page_continuation"
            )

        # Keep a small but valid region. A nearly-zero interval is a signal
        # that the start/boundary anchors collided and should not be rendered.
        if (
            bottom_y
            <= top_y
            + 12.0
        ):
            continue

        clip = fitz.Rect(
            14.0,
            top_y,
            float(
                page.rect.width
            )
            - 14.0,
            bottom_y,
        )

        image_path = (
            question_dir
            / (
                f"{filename_prefix}_{sequence:02d}_"
                f"source_{source_page}.png"
            )
        )

        render_clip_to_png(
            page=page,
            clip=clip,
            output_path=image_path,
        )

        relative_path = output_relative_path(
            image_path
        )

        image_paths.append(
            relative_path
        )

        rectangles.append(
            {
                "role": (
                    role
                    if sequence == 1
                    else (
                        role
                        + "_continuation"
                    )
                ),
                "source_page_number": int(
                    source_page
                ),
                "x0": round(
                    float(
                        clip.x0
                    ),
                    3,
                ),
                "y0": round(
                    float(
                        clip.y0
                    ),
                    3,
                ),
                "x1": round(
                    float(
                        clip.x1
                    ),
                    3,
                ),
                "y1": round(
                    float(
                        clip.y1
                    ),
                    3,
                ),
                "method": (
                    method
                    + "+"
                    + boundary_method
                ),
            }
        )

    return (
        image_paths,
        rectangles,
    )


def structural_dependency_manifest(
    *,
    question: dict[str, Any],
    required_labels: list[str],
    parent_number: str | None,
    context_paths: list[str],
    context_rectangles: list[dict[str, Any]],
    context_meta: dict[str, Any] | None,
    selected_paths: list[str],
) -> dict[str, Any]:
    """
    Populate the existing Notebook-05 manifest contract without returning to
    label-by-label geometry cropping.

    A child question's separate dependency image IS the structural parent
    context crop. A top-level question keeps dependencies inside its own crop.
    """
    normalized_labels = [
        normalized_dependency_label(
            label
        )
        for label in required_labels
    ]

    details = []

    if parent_number is None:
        resolved = (
            normalized_labels
            if selected_paths
            else []
        )

        missing = (
            []
            if selected_paths
            else normalized_labels
        )

        for label in normalized_labels:
            details.append(
                {
                    "label": label,
                    "status": (
                        "inside_top_level_selected_question_crop"
                        if selected_paths
                        else "dependency_unresolved"
                    ),
                    "strategy": (
                        "top_level_no_separate_dependency_crop"
                    ),
                }
            )

        return {
            "resolved_dependency_labels": (
                resolved
            ),
            "missing_dependency_labels": (
                missing
            ),
            "dependency_source_pages": [],
            "dependency_crop_images": [],
            "dependency_crop_rectangles": [],
            "dependency_resolution_details": (
                details
            ),
            "dependency_render_status": (
                "not_required"
                if not normalized_labels
                else (
                    "inside_selected_question"
                    if not missing
                    else "unresolved"
                )
            ),
        }

    context_ok = bool(
        context_paths
        and context_rectangles
    )

    resolved = (
        normalized_labels
        if context_ok
        else []
    )

    missing = (
        []
        if context_ok
        else normalized_labels
    )

    for label in normalized_labels:
        details.append(
            {
                "label": label,
                "status": (
                    "resolved_by_structural_parent_context"
                    if context_ok
                    else "dependency_unresolved"
                ),
                "strategy": (
                    "parent_start_to_first_child_marker"
                ),
                "parent_number": (
                    parent_number
                ),
            }
        )

    details.insert(
        0,
        {
            "status": (
                "structural_parent_context_crop"
                if context_ok
                else "structural_parent_context_unresolved"
            ),
            "parent_number": (
                parent_number
            ),
            "strategy": (
                "parent_start_or_figure_caption_to_first_child_marker"
            ),
            **(
                context_meta
                or {}
            ),
        },
    )

    dependency_pages = sorted(
        {
            int(
                rectangle[
                    "source_page_number"
                ]
            )
            for rectangle
            in context_rectangles
        }
    )

    return {
        "resolved_dependency_labels": (
            resolved
        ),
        "missing_dependency_labels": (
            missing
        ),
        "dependency_source_pages": (
            dependency_pages
        ),
        "dependency_crop_images": (
            context_paths
        ),
        "dependency_crop_rectangles": (
            context_rectangles
        ),
        "dependency_resolution_details": (
            details
        ),
        "dependency_render_status": (
            "rendered"
            if context_ok
            else "unresolved"
        ),
    }


## 5. Render selected question regions

### Child/subquestion

- derive the direct parent from `question_number`;
- locate all direct child markers of that parent on the source PDF;
- render shared dependency/context from the **parent start / required caption**
  to immediately before the **first child marker**;
- render the selected question from its **selected child marker** to immediately
  before the **next sibling marker**;
- if the selected child is the final sibling, use the next top-level question
  boundary, then content bottom only as a final fallback.

### Top-level question

- render from the selected top-level marker to the next top-level question;
- do **not** create a separate dependency crop.

If the selected start itself cannot be verified, assigned source pages are
retained only as a manual-review fallback and are not marked student-release
eligible.


In [ ]:

def render_one_question(
    question: dict[str, Any],
) -> dict[str, Any]:
    question_id = str(
        question.get(
            "question_id"
        )
        or ""
    )

    requirement = question_render_required(
        question
    )

    assigned_pages = assigned_pages_for_question(
        question
    )

    base = {
        "question_id": question_id,
        "database_visual_flag": bool(
            requirement[
                "database_visual"
            ]
        ),
        "required_dependency_labels": (
            requirement[
                "required_labels"
            ]
        ),
        "structured_response_required": bool(
            requirement[
                "structured_response_required"
            ]
        ),
        "multi_page_question": bool(
            requirement[
                "multi_page_question"
            ]
        ),
        "visual_render_required": bool(
            requirement[
                "effective_visual_required"
            ]
        ),
        "visual_render_status": (
            "not_required"
            if not requirement[
                "effective_visual_required"
            ]
            else "pending"
        ),
        "visual_dependency_complete": bool(
            not requirement[
                "effective_visual_required"
            ]
        ),
        "dependency_render_status": (
            "not_required"
        ),
        "resolved_dependency_labels": [],
        "missing_dependency_labels": [],
        "dependency_source_pages": [],
        "dependency_crop_images": [],
        "dependency_crop_rectangles": [],
        "dependency_resolution_details": [],
        "visual_render_error": None,
        "resolved_source_pdf_path": None,
        "assigned_source_pages": (
            assigned_pages
        ),
        "searched_source_pages": [],
        "source_page_numbers": [],
        "matched_source_page": None,
        "source_page_match_status": (
            "not_required"
            if not requirement[
                "effective_visual_required"
            ]
            else "pending"
        ),
        "source_page_match_score": None,
        "source_page_token_coverage": None,
        "source_page_anchor_score": None,
        "source_page_question_number_match": None,
        "final_crop_anchor_verified": bool(
            not requirement[
                "effective_visual_required"
            ]
        ),
        "structured_layout_verified": bool(
            not requirement[
                "structured_response_required"
            ]
        ),
        "rendered_page_images": [],
        "cropped_question_images": [],
        "full_page_images": [],
        "rendered_page_count": 0,
        "question_region_crop_status": (
            "not_required"
            if not requirement[
                "effective_visual_required"
            ]
            else "pending"
        ),
        "question_region_crop_method": [],
        "question_region_crop_rectangles": [],
        "student_release_eligible": bool(
            not requirement[
                "effective_visual_required"
            ]
        ),
        "renderer_version": (
            NOTEBOOK7_RENDERER_VERSION
        ),
        "dependency_resolution_version": (
            DEPENDENCY_RESOLUTION_VERSION
        ),
        "multipage_render_version": (
            MULTIPAGE_RENDER_VERSION
        ),
        # v2.1 structural diagnostics
        "structural_parent_question_number": None,
        "structural_child_markers": [],
        "structural_first_child_marker": None,
        "structural_next_sibling_marker": None,
    }

    if not requirement[
        "effective_visual_required"
    ]:
        return base

    pdf_path = resolve_local_pdf_path(
        question.get(
            "source_pdf_path"
        )
    )

    if pdf_path is None:
        base.update(
            {
                "visual_render_status": (
                    "failed"
                ),
                "visual_dependency_complete": False,
                "dependency_render_status": (
                    "failed"
                ),
                "source_page_match_status": (
                    "source_pdf_unavailable"
                ),
                "question_region_crop_status": (
                    "failed"
                ),
                "student_release_eligible": False,
                "visual_render_error": (
                    "The locally cached source Question Paper PDF "
                    "could not be resolved."
                ),
            }
        )
        return base

    base[
        "resolved_source_pdf_path"
    ] = str(
        pdf_path
    )

    source_pdf_registry_key = (
        normalized_pdf_registry_key(
            pdf_path
        )
    )

    question_dir = (
        IMAGE_ROOT
        / question_id
    )

    question_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    try:
        with fitz.open(
            str(
                pdf_path
            )
        ) as document:
            candidate_pages = bounded_candidate_pages(
                assigned_pages=assigned_pages,
                page_count=document.page_count,
            )

            base[
                "searched_source_pages"
            ] = candidate_pages

            selected_row = (
                inventory_row_for_selected_question(
                    question
                )
            )

            preferred_pages = list(
                dict.fromkeys(
                    assigned_pages
                    + (
                        inventory_row_pages(
                            selected_row
                        )
                        if selected_row is not None
                        else []
                    )
                    + candidate_pages
                )
            )

            selected_position = locate_question_position(
                document=document,
                question_number=question.get(
                    "question_number"
                ),
                question_text=question.get(
                    "question_text"
                ),
                preferred_pages=preferred_pages,
            )

            # ------------------------------------------------------------
            # Safe legacy anchor fallback: only used when the structural
            # marker/text locator could not establish the selected start.
            # ------------------------------------------------------------
            if selected_position is None:
                evaluated = []

                for source_page in candidate_pages:
                    page_index = source_number_to_index(
                        source_page
                    )

                    if (
                        page_index < 0
                        or page_index
                        >= document.page_count
                    ):
                        continue

                    page = document.load_page(
                        page_index
                    )

                    metrics = page_match_metrics(
                        page=page,
                        question=question,
                    )

                    if metrics[
                        "valid_match"
                    ]:
                        evaluated.append(
                            (
                                -float(
                                    metrics[
                                        "combined_score"
                                    ]
                                ),
                                int(
                                    source_page
                                ),
                                metrics,
                            )
                        )

                if evaluated:
                    evaluated.sort(
                        key=lambda item: (
                            item[0],
                            item[1],
                        )
                    )

                    _, source_page, metrics = (
                        evaluated[0]
                    )

                    selected_position = {
                        "question_number": str(
                            question.get(
                                "question_number"
                            )
                            or ""
                        ),
                        "source_page_number": int(
                            source_page
                        ),
                        "rect": metrics[
                            "anchor_rect"
                        ],
                        "anchor_method": (
                            "legacy_verified_text_anchor_fallback"
                        ),
                        "anchor_score": float(
                            metrics[
                                "anchor_score"
                            ]
                        ),
                    }

            # ------------------------------------------------------------
            # No verified selected start -> retain assigned pages only for
            # manual review. Do not claim structural correctness.
            # ------------------------------------------------------------
            if selected_position is None:
                fallback_pages = [
                    page_number
                    for page_number
                    in assigned_pages
                    if (
                        0
                        <= source_number_to_index(
                            page_number
                        )
                        < document.page_count
                    )
                ]

                if not fallback_pages:
                    base.update(
                        {
                            "visual_render_status": (
                                "failed"
                            ),
                            "visual_dependency_complete": False,
                            "dependency_render_status": (
                                "failed"
                            ),
                            "source_page_match_status": (
                                "question_anchor_not_found"
                            ),
                            "question_region_crop_status": (
                                "anchor_verification_failed"
                            ),
                            "student_release_eligible": False,
                            "visual_render_error": (
                                "The selected question marker/text anchor "
                                "could not be located on the source PDF."
                            ),
                        }
                    )
                    return base

                fallback_images = []
                fallback_rectangles = []

                for sequence, source_page in enumerate(
                    fallback_pages,
                    start=1,
                ):
                    page = document.load_page(
                        source_number_to_index(
                            source_page
                        )
                    )

                    clip = safe_full_content_rect(
                        page
                    )

                    image_path = (
                        question_dir
                        / (
                            f"fallback_page_{sequence:02d}_"
                            f"source_{source_page}.png"
                        )
                    )

                    render_clip_to_png(
                        page=page,
                        clip=clip,
                        output_path=image_path,
                    )

                    relative_path = output_relative_path(
                        image_path
                    )

                    fallback_images.append(
                        relative_path
                    )

                    fallback_rectangles.append(
                        {
                            "role": (
                                "safe_full_page_fallback"
                            ),
                            "source_page_number": int(
                                source_page
                            ),
                            "x0": float(
                                clip.x0
                            ),
                            "y0": float(
                                clip.y0
                            ),
                            "x1": float(
                                clip.x1
                            ),
                            "y1": float(
                                clip.y1
                            ),
                            "method": (
                                "assigned_page_safe_fallback"
                            ),
                        }
                    )

                base.update(
                    {
                        "source_page_numbers": (
                            fallback_pages
                        ),
                        "rendered_page_images": (
                            fallback_images
                        ),
                        "full_page_images": (
                            fallback_images
                        ),
                        "rendered_page_count": int(
                            len(
                                fallback_images
                            )
                        ),
                        "matched_source_page": (
                            fallback_pages[0]
                        ),
                        "source_page_match_status": (
                            "assigned_pages_safe_fallback"
                        ),
                        "final_crop_anchor_verified": False,
                        "structured_layout_verified": False,
                        "visual_dependency_complete": False,
                        "dependency_render_status": (
                            "unresolved"
                        ),
                        "visual_render_status": (
                            "rendered_fallback_requires_review"
                        ),
                        "question_region_crop_status": (
                            "full_page_fallback"
                        ),
                        "question_region_crop_method": [
                            "assigned_page_safe_fallback"
                        ],
                        "question_region_crop_rectangles": (
                            fallback_rectangles
                        ),
                        "student_release_eligible": False,
                        "visual_render_error": (
                            "Structural parent/sibling boundaries could not "
                            "be verified; assigned source pages were retained "
                            "for manual review."
                        ),
                    }
                )

                return base

            matched_page = int(
                selected_position[
                    "source_page_number"
                ]
            )

            base[
                "matched_source_page"
            ] = matched_page

            base[
                "final_crop_anchor_verified"
            ] = True

            anchor_method = str(
                selected_position.get(
                    "anchor_method"
                )
                or ""
            )

            base[
                "source_page_match_status"
            ] = (
                "verified_structural_question_marker"
                if "structural_question_marker"
                in anchor_method
                else (
                    "verified_question_anchor_fallback"
                )
            )

            # Keep the old quality diagnostics for Notebook 05 / manual review.
            matched_page_obj = document.load_page(
                source_number_to_index(
                    matched_page
                )
            )

            metrics = page_match_metrics(
                page=matched_page_obj,
                question=question,
            )

            base[
                "source_page_match_score"
            ] = float(
                metrics[
                    "combined_score"
                ]
            )

            base[
                "source_page_token_coverage"
            ] = float(
                metrics[
                    "token_coverage"
                ]
            )

            base[
                "source_page_anchor_score"
            ] = float(
                metrics[
                    "anchor_score"
                ]
            )

            base[
                "source_page_question_number_match"
            ] = bool(
                metrics[
                    "question_number_match"
                ]
            )

            parent_number = (
                structural_parent_question_number(
                    question.get(
                        "question_number"
                    )
                )
            )

            base[
                "structural_parent_question_number"
            ] = parent_number

            context_paths = []
            context_rectangles = []
            context_meta = None
            child_hits = []
            first_child = None
            next_sibling = None

            selected_number_parts = (
                question_number_parts(
                    question.get(
                        "question_number"
                    )
                )
            )

            # ============================================================
            # CHILD QUESTION:
            # parent/context -> FIRST child
            # selected child -> NEXT sibling
            # ============================================================
            if parent_number is not None:
                child_hits = locate_structural_child_markers(
                    document=document,
                    question=question,
                    parent_number=parent_number,
                )

                # Prefer the child-marker hit for the selected number so the
                # selected crop begins at exactly the same structural system
                # used for sibling boundaries.
                selected_child_hits = [
                    hit
                    for hit in child_hits
                    if (
                        question_number_parts(
                            hit.get(
                                "question_number"
                            )
                        )
                        == selected_number_parts
                    )
                ]

                if selected_child_hits:
                    selected_position = (
                        selected_child_hits[
                            0
                        ]
                    )

                    matched_page = int(
                        selected_position[
                            "source_page_number"
                        ]
                    )

                    base[
                        "matched_source_page"
                    ] = matched_page

                if not child_hits:
                    child_hits = [
                        selected_position
                    ]

                child_hits.sort(
                    key=source_position_key
                )

                first_child = child_hits[
                    0
                ]

                next_candidates = [
                    hit
                    for hit in child_hits
                    if (
                        source_position_key(
                            hit
                        )
                        > source_position_key(
                            selected_position
                        )
                    )
                ]

                next_sibling = (
                    next_candidates[
                        0
                    ]
                    if next_candidates
                    else None
                )

                (
                    context_start,
                    context_meta,
                ) = locate_parent_context_start(
                    document=document,
                    question=question,
                    parent_number=parent_number,
                    first_child=first_child,
                    required_labels=(
                        requirement[
                            "required_labels"
                        ]
                    ),
                )

                if (
                    context_start is not None
                    and position_is_before(
                        context_start,
                        first_child,
                    )
                ):
                    (
                        context_paths,
                        context_rectangles,
                    ) = render_structural_interval(
                        document=document,
                        start_position=(
                            context_start
                        ),
                        end_position=(
                            first_child
                        ),
                        fallback_last_page=int(
                            first_child[
                                "source_page_number"
                            ]
                        ),
                        question_dir=(
                            question_dir
                        ),
                        filename_prefix=(
                            "dependency_context"
                        ),
                        role=(
                            "structural_parent_context"
                        ),
                        method=(
                            "parent_start_or_figure_caption"
                            "_to_first_child_marker"
                        ),
                    )

                selected_end = (
                    next_sibling
                    if next_sibling is not None
                    else (
                        locate_next_top_level_boundary(
                            document=document,
                            question=question,
                            after_position=(
                                selected_position
                            ),
                        )
                    )
                )

                selected_method = (
                    "selected_child_marker_to_next_sibling_marker"
                    if next_sibling is not None
                    else (
                        "selected_child_marker_to_next_top_level_boundary"
                        if selected_end is not None
                        else (
                            "selected_child_marker_to_content_bottom"
                        )
                    )
                )

            # ============================================================
            # TOP-LEVEL QUESTION:
            # selected top-level marker -> next top-level marker.
            # No separate dependency/context crop.
            # ============================================================
            else:
                selected_end = (
                    locate_next_top_level_boundary(
                        document=document,
                        question=question,
                        after_position=(
                            selected_position
                        ),
                    )
                )

                selected_method = (
                    "top_level_selected_marker_to_next_top_level_boundary"
                    if selected_end is not None
                    else (
                        "top_level_selected_marker_to_content_bottom"
                    )
                )

            fallback_last_page = (
                max(
                    [
                        page
                        for page
                        in assigned_pages
                        if page
                        >= int(
                            selected_position[
                                "source_page_number"
                            ]
                        )
                    ],
                    default=int(
                        selected_position[
                            "source_page_number"
                        ]
                    ),
                )
            )

            (
                question_crop_paths,
                question_rectangles,
            ) = render_structural_interval(
                document=document,
                start_position=(
                    selected_position
                ),
                end_position=(
                    selected_end
                ),
                fallback_last_page=(
                    fallback_last_page
                ),
                question_dir=(
                    question_dir
                ),
                filename_prefix=(
                    "question"
                ),
                role=(
                    "selected_question"
                ),
                method=(
                    selected_method
                ),
            )

            if not question_crop_paths:
                base.update(
                    {
                        "visual_render_status": (
                            "failed"
                        ),
                        "visual_dependency_complete": False,
                        "dependency_render_status": (
                            "failed"
                        ),
                        "question_region_crop_status": (
                            "failed"
                        ),
                        "student_release_eligible": False,
                        "visual_render_error": (
                            "The structural selected-question interval "
                            "collapsed or could not be rendered."
                        ),
                    }
                )
                return base

            dependency_result = (
                structural_dependency_manifest(
                    question=question,
                    required_labels=(
                        requirement[
                            "required_labels"
                        ]
                    ),
                    parent_number=(
                        parent_number
                    ),
                    context_paths=(
                        context_paths
                    ),
                    context_rectangles=(
                        context_rectangles
                    ),
                    context_meta=(
                        context_meta
                    ),
                    selected_paths=(
                        question_crop_paths
                    ),
                )
            )

            dependency_complete = bool(
                not dependency_result[
                    "missing_dependency_labels"
                ]
                and (
                    parent_number is None
                    or bool(
                        context_paths
                    )
                )
            )

            combined_images = list(
                dependency_result[
                    "dependency_crop_images"
                ]
            )

            for path_value in question_crop_paths:
                if (
                    path_value
                    not in combined_images
                ):
                    combined_images.append(
                        path_value
                    )

            selected_source_pages = sorted(
                {
                    int(
                        rectangle[
                            "source_page_number"
                        ]
                    )
                    for rectangle
                    in question_rectangles
                }
            )

            multi_page = bool(
                len(
                    selected_source_pages
                )
                > 1
            )

            if dependency_complete:
                crop_status = (
                    "multi_page_cropped_verified_with_dependencies"
                    if multi_page
                    else (
                        "cropped_verified_with_dependencies"
                    )
                )
            else:
                crop_status = (
                    "required_dependency_unresolved"
                )

            base[
                "structural_child_markers"
            ] = [
                {
                    "question_number": str(
                        hit.get(
                            "question_number"
                        )
                        or ""
                    ),
                    "source_page_number": int(
                        hit[
                            "source_page_number"
                        ]
                    ),
                    "y0": round(
                        float(
                            hit[
                                "rect"
                            ].y0
                        ),
                        3,
                    ),
                    "anchor_method": str(
                        hit.get(
                            "anchor_method"
                        )
                        or ""
                    ),
                }
                for hit in child_hits
            ]

            if first_child is not None:
                base[
                    "structural_first_child_marker"
                ] = {
                    "question_number": str(
                        first_child.get(
                            "question_number"
                        )
                        or ""
                    ),
                    "source_page_number": int(
                        first_child[
                            "source_page_number"
                        ]
                    ),
                    "y0": round(
                        float(
                            first_child[
                                "rect"
                            ].y0
                        ),
                        3,
                    ),
                }

            if next_sibling is not None:
                base[
                    "structural_next_sibling_marker"
                ] = {
                    "question_number": str(
                        next_sibling.get(
                            "question_number"
                        )
                        or ""
                    ),
                    "source_page_number": int(
                        next_sibling[
                            "source_page_number"
                        ]
                    ),
                    "y0": round(
                        float(
                            next_sibling[
                                "rect"
                            ].y0
                        ),
                        3,
                    ),
                }

            base.update(
                {
                    "source_page_numbers": (
                        selected_source_pages
                    ),
                    "rendered_page_images": (
                        combined_images
                    ),
                    "cropped_question_images": (
                        question_crop_paths
                    ),
                    "full_page_images": [],
                    "rendered_page_count": int(
                        len(
                            combined_images
                        )
                    ),
                    "structured_layout_verified": True,
                    **dependency_result,
                    "visual_dependency_complete": (
                        dependency_complete
                    ),
                    "visual_render_status": (
                        "rendered_dependency_complete"
                        if dependency_complete
                        else "failed"
                    ),
                    "question_region_crop_status": (
                        crop_status
                    ),
                    "question_region_crop_method": [
                        (
                            "structural_parent_sibling_boundaries"
                            if parent_number is not None
                            else (
                                "top_level_structural_boundary"
                            )
                        )
                    ],
                    "question_region_crop_rectangles": (
                        question_rectangles
                    ),
                    "student_release_eligible": (
                        dependency_complete
                    ),
                    "visual_render_error": (
                        None
                        if dependency_complete
                        else (
                            "The selected question crop was created, but the "
                            "required structural parent/context crop could "
                            "not be verified."
                        )
                    ),
                }
            )

            register_rendered_question_rectangles(
                source_pdf_key=(
                    source_pdf_registry_key
                ),
                question=question,
                rectangles=(
                    question_rectangles
                ),
            )

            return base

    except Exception as error:
        base.update(
            {
                "visual_render_status": (
                    "failed"
                ),
                "visual_dependency_complete": False,
                "dependency_render_status": (
                    "failed"
                ),
                "question_region_crop_status": (
                    "failed"
                ),
                "source_page_match_status": (
                    "render_exception"
                ),
                "student_release_eligible": False,
                "visual_render_error": (
                    f"{type(error).__name__}: "
                    f"{error}"
                ),
            }
        )
        return base


render_questions = [
    question
    for question in QUESTIONS
    if isinstance(
        question,
        dict,
    )
]

render_records = []

for render_index, question in enumerate(
    render_questions,
    start=1,
):
    render_started = time.perf_counter()

    print(
        f"Rendering question {render_index}/{len(render_questions)}: "
        f"{question.get('question_id', '')}",
        flush=True,
    )

    render_record = render_one_question(
        question
    )

    render_records.append(
        render_record
    )

    print(
        "  -> "
        f"{render_record.get('question_region_crop_status')} "
        f"in {time.perf_counter() - render_started:.2f}s",
        flush=True,
    )

render_df = pd.DataFrame(
    render_records
)

display(
    render_df[
        [
            column
            for column in [
                "question_id",
                "visual_render_required",
                "multi_page_question",
                "structural_parent_question_number",
                "assigned_source_pages",
                "matched_source_page",
                "structural_first_child_marker",
                "structural_next_sibling_marker",
                "source_page_numbers",
                "question_region_crop_status",
                "rendered_page_images",
                "visual_render_error",
            ]
            if column
            in render_df.columns
        ]
    ]
)


## 6. Export render manifest

Notebook 05 reads the JSON manifest so list-valued fields remain real JSON
arrays instead of CSV strings. A CSV copy is also written for manual review.

In [ ]:

manifest_csv_path = (
    MANIFEST_PATH
    .with_suffix(
        ".csv"
    )
)

render_df.to_csv(
    manifest_csv_path,
    index=False,
)

successful_crop_statuses = {
    "cropped_verified_with_dependencies",
    "multi_page_cropped_verified_with_dependencies",
}

summary = {
    "renderer_version": (
        NOTEBOOK7_RENDERER_VERSION
    ),
    "crop_version": (
        QUESTION_REGION_CROP_VERSION
    ),
    "multipage_version": (
        MULTIPAGE_RENDER_VERSION
    ),
    "dependency_resolution_version": (
        DEPENDENCY_RESOLUTION_VERSION
    ),
    # v2.1 structural rules requested for Notebook 07.
    "structural_parent_from_question_number": True,
    "all_direct_child_markers_located_on_source_pdf": True,
    "dependency_context_stops_before_first_child_marker": True,
    "selected_child_stops_before_next_sibling_marker": True,
    "top_level_has_no_separate_dependency_crop": True,
    "multi_page_structural_intervals": True,
    "safe_full_page_fallback_requires_review": True,
    "run_timestamp": (
        RUN_TIMESTAMP
    ),
    "question_count": int(
        len(
            render_df
        )
    ),
    "effective_visual_questions": int(
        render_df[
            "visual_render_required"
        ].astype(
            bool
        ).sum()
        if not render_df.empty
        else 0
    ),
    "multi_page_questions": int(
        render_df[
            "multi_page_question"
        ].astype(
            bool
        ).sum()
        if not render_df.empty
        else 0
    ),
    "structural_child_questions": int(
        render_df[
            "structural_parent_question_number"
        ].notna().sum()
        if (
            not render_df.empty
            and "structural_parent_question_number"
            in render_df.columns
        )
        else 0
    ),
    "successful_question_crops": int(
        render_df[
            "question_region_crop_status"
        ].isin(
            successful_crop_statuses
        ).sum()
        if not render_df.empty
        else 0
    ),
    "structural_context_crops_created": int(
        render_df[
            "dependency_crop_images"
        ].map(
            lambda value: len(
                value
            )
            if isinstance(
                value,
                list,
            )
            else 0
        ).sum()
        if not render_df.empty
        else 0
    ),
    "missing_dependency_labels": int(
        render_df[
            "missing_dependency_labels"
        ].map(
            lambda value: len(
                value
            )
            if isinstance(
                value,
                list,
            )
            else 0
        ).sum()
        if not render_df.empty
        else 0
    ),
    "safe_full_page_fallbacks": int(
        render_df[
            "question_region_crop_status"
        ].eq(
            "full_page_fallback"
        ).sum()
        if not render_df.empty
        else 0
    ),
    "render_failures": int(
        render_df[
            "visual_render_status"
        ].eq(
            "failed"
        ).sum()
        if not render_df.empty
        else 0
    ),
    "student_release_eligible": int(
        render_df[
            "student_release_eligible"
        ].astype(
            bool
        ).sum()
        if not render_df.empty
        else 0
    ),
    "total_images_created": int(
        render_df[
            "rendered_page_count"
        ].fillna(
            0
        ).sum()
        if not render_df.empty
        else 0
    ),
}

manifest_payload = {
    "schema_version": (
        "agent2-question-render-manifest-v1.0.0"
    ),
    "generated_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "request_path": str(
        REQUEST_PATH.resolve()
    ),
    "image_root": str(
        IMAGE_ROOT.resolve()
    ),
    "csv_path": str(
        manifest_csv_path.resolve()
    ),
    "summary": summary,
    "records": json_safe(
        render_records
    ),
}

MANIFEST_PATH.write_text(
    json.dumps(
        manifest_payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(
    "Render manifest:",
    MANIFEST_PATH,
)

print(
    "CSV copy:",
    manifest_csv_path,
)

display(
    pd.DataFrame(
        [
            summary
        ]
    )
)

if DISPLAY_RENDERED_IMAGES:
    for record in render_records:
        for image_value in record.get(
            "rendered_page_images",
            [],
        ):
            image_path = (
                OUTPUT_DIR
                / image_value
            )

            if image_path.is_file():
                display(
                    IPythonImage(
                        filename=str(
                            image_path
                        )
                    )
                )


## 7. Completion checks

In [ ]:
if not MANIFEST_PATH.is_file():
    raise RuntimeError(
        "Notebook 07 did not create its render manifest."
    )

if len(render_records) != len(
    [
        question
        for question in QUESTIONS
        if isinstance(
            question,
            dict,
        )
    ]
):
    raise RuntimeError(
        "Notebook 07 render manifest row count does not match "
        "the request question count."
    )

missing_images = []

for record in render_records:
    all_record_images = []

    for image_field in (
        "dependency_crop_images",
        "cropped_question_images",
        "full_page_images",
        "rendered_page_images",
    ):
        for relative_path in record.get(
            image_field,
            [],
        ):
            if relative_path not in all_record_images:
                all_record_images.append(
                    relative_path
                )

    for relative_path in all_record_images:
        image_path = (
            OUTPUT_DIR
            / relative_path
        )

        if not image_path.is_file():
            missing_images.append(
                str(image_path)
            )

if missing_images:
    raise RuntimeError(
        "Notebook 07 manifest references missing image files: "
        + ", ".join(
            missing_images[:10]
        )
    )

print(
    "Notebook 07 completed successfully."
)